In [0]:
import pandas as pd
import numpy as np

base = "/Volumes/workspace/default/foodtohealth/"

# === LOAD EVERYTHING (corrected file names) ===
hno_24 = pd.read_csv(f"{base}hpc_hno_2024.csv", low_memory=False)
hno_25 = pd.read_csv(f"{base}hpc_hno_2025.csv", low_memory=False)
hno_26 = pd.read_csv(f"{base}hpc_hno_2026.csv", low_memory=False)
fund_cluster = pd.read_csv(f"{base}fts_requirements_funding_cluster_global.csv")
fund_global = pd.read_csv(f"{base}fts_requirements_funding_global.csv")
fund_globalcluster = pd.read_csv(f"{base}fts_requirements_funding_globalcluster_global.csv")
fund_covid = pd.read_csv(f"{base}fts_requirements_funding_covid_global.csv")
incoming = pd.read_csv(f"{base}fts_incoming_funding_global.csv")
outgoing = pd.read_csv(f"{base}fts_outgoing_funding_global.csv")
pop_0 = pd.read_csv(f"{base}cod_population_admin0.csv")
pop_1 = pd.read_csv(f"{base}cod_population_admin1.csv")
pop_4 = pd.read_csv(f"{base}cod_population_admin4.csv")
hrp = pd.read_csv(f"{base}humanitarian-response-plans.csv")
allocations = pd.read_csv(f"{base}Allocations__20260221_193441_UTC.csv")
contributions = pd.read_csv(f"{base}Contributions__20260221_193441_UTC.csv")
metadata_hno = pd.read_csv(f"{base}metadata-global-hpc-hno-2025.csv")

# === FIX NUMERIC TYPES ===
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    for df in [hno_24, hno_25, hno_26]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'percentFunded']:
    for df in [fund_cluster, fund_global, fund_globalcluster, fund_covid]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

for df in [pop_0, pop_1, pop_4]:
    df['Population'] = pd.to_numeric(df['Population'], errors='coerce')

incoming['amountUSD'] = pd.to_numeric(incoming['amountUSD'], errors='coerce')
if 'amountUSD' in outgoing.columns:
    outgoing['amountUSD'] = pd.to_numeric(outgoing['amountUSD'], errors='coerce')

# === PROFILE EVERYTHING ===
all_datasets = {
    "hno_24": hno_24, "hno_25": hno_25, "hno_26": hno_26,
    "fund_cluster": fund_cluster, "fund_global": fund_global,
    "fund_globalcluster": fund_globalcluster, "fund_covid": fund_covid,
    "incoming": incoming, "outgoing": outgoing,
    "pop_0": pop_0, "pop_1": pop_1, "pop_4": pop_4,
    "hrp": hrp, "allocations": allocations, "contributions": contributions,
    "metadata_hno": metadata_hno
}

print("=" * 80)
print("FULL DATA INVENTORY — 17 DATASETS")
print("=" * 80)
for name, df in all_datasets.items():
    print(f"\n--- {name}: {df.shape[0]} rows x {df.shape[1]} cols ---")
    print(f"  Columns: {df.columns.tolist()}")

# === JOIN KEY ANALYSIS ===
print("\n" + "=" * 80)
print("JOIN KEY ANALYSIS — Country identifiers across all datasets")
print("=" * 80)
for name, df in all_datasets.items():
    country_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['iso3', 'country', 'location', 'countrycode'])]
    if country_cols:
        for cc in country_cols:
            unique = df[cc].nunique()
            sample = df[cc].dropna().astype(str).unique()[:5]
            print(f"  {name}.{cc}: {unique} unique | sample: {sample}")

# === CLUSTERS IN EVERY FUNDING DATASET ===
print("\n" + "=" * 80)
print("CLUSTERS/SECTORS ACROSS ALL FUNDING DATA")
print("=" * 80)
for name in ['fund_cluster', 'fund_globalcluster', 'incoming', 'outgoing']:
    df = all_datasets[name]
    cluster_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['cluster', 'sector'])]
    for cc in cluster_cols:
        # Show health and food related entries
        health_matches = df[df[cc].astype(str).str.contains('ealth', na=False, case=False)]
        food_matches = df[df[cc].astype(str).str.contains('ood|nutrition|FSC', na=False, case=False)]
        print(f"\n  {name}.{cc}:")
        print(f"    Total rows: {len(df)}")
        print(f"    Health-related rows: {len(health_matches)}")
        print(f"    Food/Nutrition-related rows: {len(food_matches)}")
        print(f"    Top 10 values:")
        print(f"    {df[cc].value_counts().head(10).to_string()}")

# === TEMPORAL COVERAGE ===
print("\n" + "=" * 80)
print("TEMPORAL COVERAGE")
print("=" * 80)
for name, df in all_datasets.items():
    year_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['year', 'date'])]
    for yc in year_cols:
        sample = df[yc].dropna().astype(str).unique()[:10]
        print(f"  {name}.{yc}: {df[yc].nunique()} unique | sample: {sample}")

# === ALLOCATIONS & CONTRIBUTIONS (small but important) ===
print("\n" + "=" * 80)
print("ALLOCATIONS (Pooled Fund)")
print("=" * 80)
print(allocations.to_string())

print("\n" + "=" * 80)
print("CONTRIBUTIONS (Pooled Fund)")
print("=" * 80)
print(contributions.head(20).to_string())

# === HNO CLUSTER COUNTS ===
print("\n" + "=" * 80)
print("HNO HEALTH vs FOOD NEEDS BY YEAR")
print("=" * 80)
for year, df in [("2024", hno_24), ("2025", hno_25)]:
    print(f"\n  {year}:")
    for code, label in [('HEA','Health'), ('FSC','Food Security'), ('NUT','Nutrition'), 
                         ('WSH','WASH'), ('SHL','Shelter'), ('PRO','Protection'), 
                         ('EDU','Education'), ('ALL','Cross-sector')]:
        sub = df[df['Cluster'] == code]
        n_countries = sub['Country ISO3'].nunique()
        need = sub['In Need'].sum()
        print(f"    {label:15s} ({code}): {n_countries:3d} countries, {need:>15,.0f} in need")

# === OUTGOING STRUCTURE ===
print("\n" + "=" * 80)
print("OUTGOING FUNDING STRUCTURE")
print("=" * 80)
print(f"Columns: {outgoing.columns.tolist()}")
print(outgoing.head(5).to_string())

print("\n\n=== ALL 17 DATASETS LOADED AND PROFILED ===")

In [0]:
import pandas as pd
import numpy as np

base = "/Volumes/workspace/default/foodtohealth/"

# ============================================================
# STEP 1: RELOAD & REMOVE HXL TAG ROWS
# ============================================================
hno_24 = pd.read_csv(f"{base}hpc_hno_2024.csv", low_memory=False)
hno_25 = pd.read_csv(f"{base}hpc_hno_2025.csv", low_memory=False)
fund_globalcluster = pd.read_csv(f"{base}fts_requirements_funding_globalcluster_global.csv")
fund_global = pd.read_csv(f"{base}fts_requirements_funding_global.csv")
fund_cluster = pd.read_csv(f"{base}fts_requirements_funding_cluster_global.csv")
fund_covid = pd.read_csv(f"{base}fts_requirements_funding_covid_global.csv")
incoming = pd.read_csv(f"{base}fts_incoming_funding_global.csv")
outgoing = pd.read_csv(f"{base}fts_outgoing_funding_global.csv")
pop_0 = pd.read_csv(f"{base}cod_population_admin0.csv")
pop_1 = pd.read_csv(f"{base}cod_population_admin1.csv")
hrp = pd.read_csv(f"{base}humanitarian-response-plans.csv")
allocations = pd.read_csv(f"{base}Allocations__20260221_193441_UTC.csv")
contributions = pd.read_csv(f"{base}Contributions__20260221_193441_UTC.csv")

# REMOVE HXL TAG ROWS (first row starting with '#')
for df in [hno_24, hno_25, fund_globalcluster, fund_global, fund_cluster, 
           fund_covid, incoming, outgoing, hrp]:
    first_val = str(df.iloc[0, 0])
    if first_val.startswith('#'):
        df.drop(0, inplace=True)
        df.reset_index(drop=True, inplace=True)

# FIX NUMERIC TYPES
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    hno_24[col] = pd.to_numeric(hno_24[col], errors='coerce')
    hno_25[col] = pd.to_numeric(hno_25[col], errors='coerce')

for col in ['requirements', 'funding', 'percentFunded']:
    for df in [fund_globalcluster, fund_global, fund_cluster]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'covidFunding']:
    if col in fund_covid.columns:
        fund_covid[col] = pd.to_numeric(fund_covid[col], errors='coerce')

incoming['amountUSD'] = pd.to_numeric(incoming['amountUSD'], errors='coerce')
outgoing['amountUSD'] = pd.to_numeric(outgoing['amountUSD'], errors='coerce')
pop_0['Population'] = pd.to_numeric(pop_0['Population'], errors='coerce')
pop_1['Population'] = pd.to_numeric(pop_1['Population'], errors='coerce')

for col in ['origRequirements', 'revisedRequirements']:
    hrp[col] = pd.to_numeric(hrp[col], errors='coerce')

allocations['Budget'] = pd.to_numeric(allocations['Budget'], errors='coerce')
for col in ['Paid', 'Pledged', 'Total']:
    contributions[col] = pd.to_numeric(contributions[col], errors='coerce')

print("Step 1 done: All 13 datasets loaded, HXL rows removed, types fixed.")
print(f"  HNO 2024: {len(hno_24)} rows, countries: {hno_24['Country ISO3'].nunique()}")
print(f"  HNO 2025: {len(hno_25)} rows, countries: {hno_25['Country ISO3'].nunique()}")
print(f"  Fund GlobalCluster: {len(fund_globalcluster)} rows")
print(f"  Fund Global: {len(fund_global)} rows")

# ============================================================
# STEP 2: BUILD HEALTH NEEDS TABLE (HNO)
# ============================================================
health_24 = hno_24[hno_24['Cluster'] == 'HEA'].groupby('Country ISO3').agg(
    health_in_need_24=('In Need', 'sum'),
    health_targeted_24=('Targeted', 'sum'),
    health_affected_24=('Affected', 'sum'),
    health_reached_24=('Reached', 'sum')
).reset_index()

health_25 = hno_25[hno_25['Cluster'] == 'HEA'].groupby('Country ISO3').agg(
    health_in_need_25=('In Need', 'sum'),
    health_targeted_25=('Targeted', 'sum')
).reset_index()

food_24 = hno_24[hno_24['Cluster'] == 'FSC'].groupby('Country ISO3').agg(
    food_in_need_24=('In Need', 'sum')
).reset_index()

food_25 = hno_25[hno_25['Cluster'] == 'FSC'].groupby('Country ISO3').agg(
    food_in_need_25=('In Need', 'sum')
).reset_index()

# ALL clusters aggregated (total needs)
all_needs_24 = hno_24.groupby('Country ISO3').agg(
    total_in_need_24=('In Need', 'sum'),
    total_population_24=('Population', 'sum')
).reset_index()

print(f"\nStep 2 done: Health needs in {len(health_24)} countries (2024), {len(health_25)} (2025)")

# ============================================================
# STEP 3: BUILD FUNDING TABLE (fund_globalcluster — BEST SOURCE)
# ============================================================

# Health funding across all years
health_funding = fund_globalcluster[
    fund_globalcluster['cluster'].str.contains('^Health$', na=False, regex=True)
].groupby(['countryCode', 'year']).agg(
    health_required=('requirements', 'sum'),
    health_funded=('funding', 'sum')
).reset_index()

# Food Security funding
food_funding = fund_globalcluster[
    fund_globalcluster['cluster'] == 'Food Security'
].groupby(['countryCode', 'year']).agg(
    food_required=('requirements', 'sum'),
    food_funded=('funding', 'sum')
).reset_index()

# ALL clusters total funding per country per year
total_funding = fund_global.groupby(['countryCode', 'year']).agg(
    total_required=('requirements', 'sum'),
    total_funded=('funding', 'sum')
).reset_index()

# Get latest year per country
health_funding_latest = health_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')
food_funding_latest = food_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')
total_funding_latest = total_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')

print(f"Step 3 done: Health funding for {health_funding_latest['countryCode'].nunique()} countries")
print(f"  Food funding for {food_funding_latest['countryCode'].nunique()} countries")
print(f"  Total funding for {total_funding_latest['countryCode'].nunique()} countries")

# ============================================================
# STEP 4: POPULATION TABLE
# ============================================================
total_pop = pop_0.groupby('ISO3')['Population'].sum().reset_index().rename(columns={'Population': 'total_pop'})
print(f"\nStep 4 done: Population for {len(total_pop)} countries")

# ============================================================
# STEP 5: COVID FUNDING
# ============================================================
covid_funding = fund_covid.groupby('countryCode').agg(
    covid_total_funding=('funding', 'sum'),
    covid_specific_funding=('covidFunding', 'sum')
).reset_index()
print(f"Step 5 done: COVID funding for {len(covid_funding)} countries")

# ============================================================
# STEP 6: INCOMING DONOR FLOWS — Health-directed funding by source
# ============================================================
health_incoming = incoming[incoming['destGlobalClusters'].str.contains('Health', na=False, case=False)]
health_donor_summary = health_incoming.groupby('srcOrganization')['amountUSD'].sum().sort_values(ascending=False)
print(f"\nStep 6 done: {len(health_incoming)} health-directed incoming flows")
print(f"  Top 5 health donors:")
print(health_donor_summary.head(5).to_string())

# ============================================================
# STEP 7: POOLED FUND ALLOCATIONS
# ============================================================
alloc_by_country = allocations.groupby('PooledFund').agg(
    total_alloc=('Budget', 'sum'),
    n_allocations=('Budget', 'count')
).sort_values('total_alloc', ascending=False).reset_index()
print(f"\nStep 7 done: Pooled fund allocations for {len(alloc_by_country)} funds")
print(alloc_by_country.to_string())

# ============================================================
# STEP 8: THE MASTER JOIN — ALL DATASETS CONNECTED
# ============================================================
master = health_24.merge(health_25, on='Country ISO3', how='outer')
master = master.merge(food_24, on='Country ISO3', how='outer')
master = master.merge(food_25, on='Country ISO3', how='outer')
master = master.merge(all_needs_24, on='Country ISO3', how='outer')
master = master.merge(health_funding_latest[['countryCode', 'health_required', 'health_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left')
master = master.merge(food_funding_latest[['countryCode', 'food_required', 'food_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_food'))
master = master.merge(total_funding_latest[['countryCode', 'total_required', 'total_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_total'))
master = master.merge(total_pop, left_on='Country ISO3', right_on='ISO3', how='left')
master = master.merge(covid_funding, left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_covid'))

# Drop redundant join columns
master = master.drop(columns=[c for c in master.columns if c.startswith('countryCode') or c == 'ISO3'], errors='ignore')

# ============================================================
# STEP 9: COMPUTE NEW METRICS
# ============================================================

# Funding gaps
master['health_gap'] = master['health_required'] - master['health_funded']
master['food_gap'] = master['food_required'] - master['food_funded']
master['total_gap'] = master['total_required'] - master['total_funded']

# Coverage ratios
master['health_coverage'] = master['health_funded'] / master['health_required']
master['food_coverage'] = master['food_funded'] / master['food_required']
master['total_coverage'] = master['total_funded'] / master['total_required']

# Per-capita metrics
master['health_funding_per_person_in_need'] = master['health_funded'] / master['health_in_need_24']
master['health_gap_per_person_in_need'] = master['health_gap'] / master['health_in_need_24']
master['total_funding_per_capita'] = master['total_funded'] / master['total_pop']

# Health share of total funding
master['health_share_of_funding'] = master['health_funded'] / master['total_funded']
master['health_share_of_requirements'] = master['health_required'] / master['total_required']

# Health vs Food comparison
master['health_vs_food_coverage_gap'] = master['food_coverage'] - master['health_coverage']

# Year-over-year need change
master['health_need_change_pct'] = (
    (master['health_in_need_25'] - master['health_in_need_24']) / master['health_in_need_24'] * 100
)
master['food_need_change_pct'] = (
    (master['food_in_need_25'] - master['food_in_need_24']) / master['food_in_need_24'] * 100
)

# Food-to-Health lag signal: countries where food needs grew in 2024 and health needs grew in 2025
master['food_to_health_lag'] = (
    (master['food_in_need_24'] > master['food_in_need_25']) &  # food needs were higher in 2024
    (master['health_in_need_25'] > master['health_in_need_24'])  # health needs grew by 2025
)

# Sort by health gap
master = master.sort_values('health_gap', ascending=False)

print(f"\nStep 9 done: Master table with {len(master)} countries, {len(master.columns)} columns")

# ============================================================
# STEP 10: DISPLAY THE FULL MISMATCH TABLE
# ============================================================
display_cols = [
    'Country ISO3', 
    'health_in_need_24', 'health_in_need_25', 'health_need_change_pct',
    'food_in_need_24', 'food_in_need_25', 'food_need_change_pct',
    'health_required', 'health_funded', 'health_gap', 'health_coverage',
    'food_required', 'food_funded', 'food_gap', 'food_coverage',
    'total_required', 'total_funded', 'total_gap', 'total_coverage',
    'health_funding_per_person_in_need', 'health_gap_per_person_in_need',
    'health_share_of_funding', 'health_vs_food_coverage_gap',
    'total_pop', 'total_funding_per_capita',
    'food_to_health_lag'
]

print("\n" + "=" * 120)
print("CRISIS COMPASS: FULL MISMATCH TABLE — ALL DATASETS CONNECTED")
print("=" * 120)
print(master[[c for c in display_cols if c in master.columns]].to_string())

# ============================================================
# STEP 11: KEY INSIGHTS SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

m = master.dropna(subset=['health_coverage'])
print(f"\n1. MOST UNDERFUNDED HEALTH CRISES (lowest coverage ratio):")
print(m.nsmallest(10, 'health_coverage')[['Country ISO3', 'health_in_need_24', 'health_coverage', 'health_gap']].to_string())

print(f"\n2. HEALTH VS FOOD DISPARITY (health more underfunded than food):")
disparity = m[m['health_vs_food_coverage_gap'] > 0].sort_values('health_vs_food_coverage_gap', ascending=False)
print(disparity[['Country ISO3', 'health_coverage', 'food_coverage', 'health_vs_food_coverage_gap']].head(10).to_string())

print(f"\n3. BIGGEST HEALTH NEED INCREASES 2024→2025:")
print(m.nlargest(10, 'health_need_change_pct')[['Country ISO3', 'health_in_need_24', 'health_in_need_25', 'health_need_change_pct']].to_string())

print(f"\n4. FOOD-TO-HEALTH LAG SIGNAL (food↑ then health↑):")
lag = master[master['food_to_health_lag'] == True]
print(f"   {len(lag)} countries show the pattern")
if len(lag) > 0:
    print(lag[['Country ISO3', 'food_in_need_24', 'food_in_need_25', 'health_in_need_24', 'health_in_need_25']].to_string())

print(f"\n5. LOWEST HEALTH FUNDING PER PERSON IN NEED:")
print(m.nsmallest(10, 'health_funding_per_person_in_need')[['Country ISO3', 'health_in_need_24', 'health_funded', 'health_funding_per_person_in_need']].to_string())

print(f"\n6. POOLED FUND ALLOCATION TOTALS:")
print(f"   Total allocated: ${allocations['Budget'].sum():,.0f}")
print(f"   Top recipients: {alloc_by_country.head(5)[['PooledFund', 'total_alloc']].to_string()}")

print(f"\n7. TOP HEALTH DONORS (incoming flows):")
print(health_donor_summary.head(10).to_string())

print(f"\n8. COVID FUNDING LEGACY:")
covid_merged = master.dropna(subset=['covid_total_funding'])
print(f"   {len(covid_merged)} countries had COVID-specific funding")
if len(covid_merged) > 0:
    print(covid_merged[['Country ISO3', 'covid_total_funding', 'covid_specific_funding', 'health_coverage']].sort_values('covid_total_funding', ascending=False).head(10).to_string())

In [0]:
# Convert numeric columns that loaded as strings
import numpy as np

# HNO — convert need/population columns to numeric
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    hno_24[col] = pd.to_numeric(hno_24[col].replace('Unknown', np.nan), errors='coerce')
    hno_25[col] = pd.to_numeric(hno_25[col].replace('Unknown', np.nan), errors='coerce')

# Fund cluster — convert funding columns to numeric
for col in ['requirements', 'funding', 'percentFunded']:
    fund_cluster[col] = pd.to_numeric(fund_cluster[col].replace('Unknown', np.nan), errors='coerce')
    fund_global[col] = pd.to_numeric(fund_global[col].replace('Unknown', np.nan), errors='coerce')

# Population
pop_0['Population'] = pd.to_numeric(pop_0['Population'], errors='coerce')
pop_1['Population'] = pd.to_numeric(pop_1['Population'], errors='coerce')

# Incoming funding
incoming['amountUSD'] = pd.to_numeric(incoming['amountUSD'].replace('Unknown', np.nan), errors='coerce')

print("Types fixed.")

In [0]:
# ============================================================
# INVISIBLE CRISIS SCORE + ABSORPTION CAPACITY (PROXY)
# ============================================================

# Filter to countries with valid health data (exclude inf)
scored = master[
    (master['health_required'] > 0) & 
    (master['health_in_need_24'].notna()) &
    (master['health_coverage'] != np.inf)
].copy()

# --- INVISIBLE CRISIS SCORE ---
# Formula: (people in need) × (1 - coverage ratio) × (1 / funding per person)
# Higher = worse = more invisible
scored['inverse_coverage'] = 1 - scored['health_coverage']
scored['inverse_per_capita'] = 1 / scored['health_funding_per_person_in_need'].replace(0, np.nan)

# Normalize components to 0-1 scale
for col in ['health_in_need_24', 'inverse_coverage', 'inverse_per_capita', 'health_gap']:
    min_val = scored[col].min()
    max_val = scored[col].max()
    scored[f'{col}_norm'] = (scored[col] - min_val) / (max_val - min_val) if max_val > min_val else 0

# Composite score (equal weighting)
scored['invisible_crisis_score'] = (
    scored['health_in_need_24_norm'] * 0.35 +    # Scale of need
    scored['inverse_coverage_norm'] * 0.35 +       # How underfunded
    scored['inverse_per_capita_norm'] * 0.30        # How little per person
)

# Rank
scored['crisis_rank'] = scored['invisible_crisis_score'].rank(ascending=False).astype(int)

print("=" * 100)
print("INVISIBLE CRISIS SCORE — RANKED")
print("=" * 100)
print(scored[['Country ISO3', 'crisis_rank', 'invisible_crisis_score',
              'health_in_need_24', 'health_coverage', 'health_funding_per_person_in_need',
              'health_gap', 'health_need_change_pct']].sort_values('crisis_rank').to_string())

# --- SENSITIVITY ANALYSIS (3 weighting schemes) ---
print("\n" + "=" * 100)
print("SENSITIVITY ANALYSIS — Do rankings change with different weights?")
print("=" * 100)

weight_schemes = {
    'Equal (35/35/30)':        (0.35, 0.35, 0.30),
    'Need-heavy (50/25/25)':   (0.50, 0.25, 0.25),
    'Coverage-heavy (25/50/25)': (0.25, 0.50, 0.25)
}

rankings = {}
for scheme_name, (w1, w2, w3) in weight_schemes.items():
    scored[f'score_{scheme_name}'] = (
        scored['health_in_need_24_norm'] * w1 +
        scored['inverse_coverage_norm'] * w2 +
        scored['inverse_per_capita_norm'] * w3
    )
    rankings[scheme_name] = scored.sort_values(f'score_{scheme_name}', ascending=False)['Country ISO3'].tolist()

# Show top 10 under each scheme
for scheme_name in weight_schemes:
    print(f"\n  {scheme_name}: {rankings[scheme_name][:10]}")

# Check stability: countries in top 5 across ALL schemes
from collections import Counter
all_top5 = []
for scheme_name in weight_schemes:
    all_top5.extend(rankings[scheme_name][:5])
stable_top5 = [c for c, count in Counter(all_top5).items() if count == len(weight_schemes)]
print(f"\n  STABLE TOP 5 (appear in every scheme): {stable_top5}")

# --- TEMPORAL ANALYSIS ---
print("\n" + "=" * 100)
print("TEMPORAL: HEALTH NEED TRAJECTORY (2024 → 2025)")
print("=" * 100)

temporal = scored[scored['health_need_change_pct'].notna()].sort_values('health_need_change_pct', ascending=False)
print("\n  WORSENING (need increasing):")
worsening = temporal[temporal['health_need_change_pct'] > 0]
print(worsening[['Country ISO3', 'health_in_need_24', 'health_in_need_25', 
                  'health_need_change_pct', 'health_coverage', 'crisis_rank']].to_string())

print("\n  IMPROVING (need decreasing):")
improving = temporal[temporal['health_need_change_pct'] < 0]
print(improving[['Country ISO3', 'health_in_need_24', 'health_in_need_25', 
                  'health_need_change_pct', 'health_coverage', 'crisis_rank']].to_string())

# --- HEALTH VS ALL OTHER SECTORS ---
print("\n" + "=" * 100)
print("HEALTH SHARE OF TOTAL FUNDING vs HEALTH SHARE OF TOTAL NEED")
print("=" * 100)

scored['health_need_share'] = scored['health_in_need_24'] / scored['total_in_need_24']
scored['funding_vs_need_share_gap'] = scored['health_share_of_funding'] - scored['health_need_share']
print(scored[['Country ISO3', 'health_need_share', 'health_share_of_funding', 
              'funding_vs_need_share_gap']].sort_values('funding_vs_need_share_gap').to_string())

# ============================================================
# EXPORT FOR FIGMA MAKE
# ============================================================

figma_export = scored[['Country ISO3', 'crisis_rank', 'invisible_crisis_score',
                        'health_in_need_24', 'health_in_need_25', 'health_need_change_pct',
                        'food_in_need_24', 
                        'health_required', 'health_funded', 'health_gap', 'health_coverage',
                        'food_required', 'food_funded', 'food_coverage',
                        'health_funding_per_person_in_need', 'health_gap_per_person_in_need',
                        'health_vs_food_coverage_gap',
                        'total_pop', 'total_funding_per_capita',
                        'food_to_health_lag']].copy()

# Round for readability
for col in figma_export.select_dtypes(include=[np.number]).columns:
    figma_export[col] = figma_export[col].round(4)

figma_export = figma_export.sort_values('crisis_rank')

# Save
figma_export.to_csv(f"{base}figma_export.csv", index=False)

# Also save the full master
master.to_csv(f"{base}crisis_compass_master.csv", index=False)

print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)
print(f"  {base}figma_export.csv — {len(figma_export)} rows (for Figma Make)")
print(f"  {base}crisis_compass_master.csv — {len(master)} rows (full analysis)")

print("\n  FIGMA EXPORT PREVIEW:")
print(figma_export.to_string())

print(f"\n  Copy this CSV for Figma Make:")
print(figma_export.to_csv(index=False))

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

# Use scored dataframe from previous cell
# If not available, reload:
# scored = pd.read_csv(f"{base}crisis_compass_master.csv")

fig = plt.figure(figsize=(24, 28))
fig.suptitle('CRISIS COMPASS: The Food–Health Nexus in Humanitarian Crises', 
             fontsize=20, fontweight='bold', y=0.98)

# ============================================================
# PLOT 1: Health vs Food Coverage Ratio (Scatter)
# Shows: Both are underfunded, but how do they compare?
# ============================================================
ax1 = fig.add_subplot(3, 2, 1)
s = scored.dropna(subset=['health_coverage', 'food_coverage'])
s = s[(s['health_coverage'] != np.inf) & (s['food_coverage'] != np.inf)]

sizes = s['health_in_need_24'] / s['health_in_need_24'].max() * 800 + 50
ax1.scatter(s['food_coverage'] * 100, s['health_coverage'] * 100, 
            s=sizes, alpha=0.7, c=s['invisible_crisis_score'], cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)

# Diagonal line (equal funding)
max_val = max(s['food_coverage'].max() * 100, s['health_coverage'].max() * 100)
ax1.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='Equal coverage')

for _, row in s.iterrows():
    ax1.annotate(row['Country ISO3'], (row['food_coverage']*100, row['health_coverage']*100),
                fontsize=7, fontweight='bold', ha='center', va='bottom')

ax1.set_xlabel('Food Security Coverage (%)', fontsize=11)
ax1.set_ylabel('Health Coverage (%)', fontsize=11)
ax1.set_title('Health vs Food: Both Underfunded, Health Worse\n(bubble size = people in health need, color = crisis score)', fontsize=12)
ax1.legend(fontsize=9)
ax1.set_xlim(0, None)
ax1.set_ylim(0, None)

# ============================================================
# PLOT 2: Paired Bar Chart — Health vs Food Funding Gap
# ============================================================
ax2 = fig.add_subplot(3, 2, 2)
top_countries = scored.nlargest(12, 'health_gap').copy()
top_countries = top_countries.sort_values('health_gap', ascending=True)

y_pos = np.arange(len(top_countries))
bar_height = 0.35

ax2.barh(y_pos + bar_height/2, top_countries['health_gap'] / 1e6, bar_height, 
         label='Health Funding Gap', color='#e74c3c', alpha=0.85)
ax2.barh(y_pos - bar_height/2, top_countries['food_gap'] / 1e6, bar_height, 
         label='Food Security Funding Gap', color='#f39c12', alpha=0.85)

ax2.set_yticks(y_pos)
ax2.set_yticklabels(top_countries['Country ISO3'], fontsize=9)
ax2.set_xlabel('Funding Gap ($ millions)', fontsize=11)
ax2.set_title('Health vs Food Security Funding Gaps\n(Side by side — same countries)', fontsize=12)
ax2.legend(fontsize=9)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))

# ============================================================
# PLOT 3: Food-to-Health Lag — Temporal Arrows
# Shows: Countries where food needs dropped but health needs rose
# ============================================================
ax3 = fig.add_subplot(3, 2, 3)
temporal = scored.dropna(subset=['health_need_change_pct', 'food_need_change_pct']).copy()

colors = ['#e74c3c' if row['food_to_health_lag'] else '#3498db' 
          for _, row in temporal.iterrows()]

ax3.scatter(temporal['food_need_change_pct'], temporal['health_need_change_pct'],
           c=colors, s=150, edgecolors='black', linewidth=0.5, zorder=5)

for _, row in temporal.iterrows():
    ax3.annotate(row['Country ISO3'], 
                (row['food_need_change_pct'], row['health_need_change_pct']),
                fontsize=8, fontweight='bold', ha='center', va='bottom')

# Quadrant lines
ax3.axhline(0, color='gray', linewidth=0.8, linestyle='-')
ax3.axvline(0, color='gray', linewidth=0.8, linestyle='-')

# Highlight the lag quadrant
ax3.fill_between([-80, 0], [0, 0], [300, 300], alpha=0.08, color='red')
ax3.text(-40, 200, 'FOOD→HEALTH LAG\nFood needs ↓, Health needs ↑', 
         fontsize=9, color='red', ha='center', style='italic', fontweight='bold')

ax3.set_xlabel('Food Security Need Change 2024→2025 (%)', fontsize=11)
ax3.set_ylabel('Health Need Change 2024→2025 (%)', fontsize=11)
ax3.set_title('The Food→Health Lag Signal\n(Red dots = food crisis converted to health crisis)', fontsize=12)

# ============================================================
# PLOT 4: $ Per Person in Need — Health vs Food
# ============================================================
ax4 = fig.add_subplot(3, 2, 4)
compare = scored.dropna(subset=['health_funding_per_person_in_need']).copy()
compare['food_funding_per_person_in_need'] = compare['food_funded'] / compare['food_in_need_24']
compare = compare.sort_values('health_funding_per_person_in_need', ascending=True).head(15)

y_pos = np.arange(len(compare))
bar_height = 0.35

ax4.barh(y_pos + bar_height/2, compare['health_funding_per_person_in_need'], bar_height,
         label='Health $ / person in need', color='#e74c3c', alpha=0.85)
ax4.barh(y_pos - bar_height/2, compare['food_funding_per_person_in_need'].clip(upper=5), bar_height,
         label='Food $ / person in need', color='#f39c12', alpha=0.85)

ax4.set_yticks(y_pos)
ax4.set_yticklabels(compare['Country ISO3'], fontsize=9)
ax4.set_xlabel('Funding per Person in Need ($)', fontsize=11)
ax4.set_title('How Much Does Each Person Receive?\n(Health vs Food — per capita)', fontsize=12)
ax4.legend(fontsize=9)

# ============================================================
# PLOT 5: Invisible Crisis Score — Ranked
# ============================================================
ax5 = fig.add_subplot(3, 2, 5)
ranked = scored.sort_values('invisible_crisis_score', ascending=True)

colors_bar = plt.cm.RdYlGn_r(ranked['invisible_crisis_score'] / ranked['invisible_crisis_score'].max())
ax5.barh(range(len(ranked)), ranked['invisible_crisis_score'], color=colors_bar, edgecolor='black', linewidth=0.3)
ax5.set_yticks(range(len(ranked)))
ax5.set_yticklabels(ranked['Country ISO3'], fontsize=9)
ax5.set_xlabel('Invisible Crisis Score', fontsize=11)
ax5.set_title('Invisible Crisis Score Rankings\n(Higher = more overlooked by humanitarian system)', fontsize=12)

# Add value labels
for i, (_, row) in enumerate(ranked.iterrows()):
    ax5.text(row['invisible_crisis_score'] + 0.01, i, f"{row['invisible_crisis_score']:.2f}", 
            va='center', fontsize=7)

# ============================================================
# PLOT 6: Stacked — Health Need vs Health Funded vs Gap
# Shows the sheer scale of underfunding
# ============================================================
ax6 = fig.add_subplot(3, 2, 6)
gap_data = scored.nlargest(12, 'health_required').sort_values('health_required', ascending=True).copy()

y_pos = np.arange(len(gap_data))
ax6.barh(y_pos, gap_data['health_funded'] / 1e6, color='#27ae60', label='Health Funded', alpha=0.85)
ax6.barh(y_pos, gap_data['health_gap'] / 1e6, left=gap_data['health_funded'] / 1e6, 
         color='#e74c3c', label='Health Funding Gap', alpha=0.85)

ax6.set_yticks(y_pos)
ax6.set_yticklabels(gap_data['Country ISO3'], fontsize=9)
ax6.set_xlabel('USD (millions)', fontsize=11)
ax6.set_title('Health Funding: What Was Asked vs What Was Received\n(Green = funded, Red = gap)', fontsize=12)
ax6.legend(fontsize=9)
ax6.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))

# Add coverage % labels
for i, (_, row) in enumerate(gap_data.iterrows()):
    pct = row['health_coverage'] * 100
    ax6.text(row['health_required'] / 1e6 + 2, i, f'{pct:.1f}%', va='center', fontsize=8, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f"{base}crisis_compass_visualizations.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {base}crisis_compass_visualizations.png")

In [0]:
import requests
import pandas as pd
import numpy as np

base = "/Volumes/workspace/default/foodtohealth/"

# Your 18 crisis countries from the scored table
countries = ['HTI', 'AFG', 'COL', 'NGA', 'UKR', 'MLI', 'BFA', 'MMR', 'COD', 
             'SSD', 'SOM', 'MOZ', 'SLV', 'HND', 'CAF', 'ETH', 'TCD', 'GTM']

country_str = ';'.join(countries)

# ============================================================
# WORLD BANK INDICATORS — What each one means
# ============================================================
indicators = {
    # GDP per capita (current US$) — measures economic capacity
    # Higher = more resources to absorb humanitarian health funding
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    
    # Current health expenditure (% of GDP) — how much the country spends on health
    # Higher = stronger existing health system to channel aid through
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure_pct_gdp',
    
    # Physicians per 1,000 people — health workforce capacity
    # Higher = more staff to deliver health services funded by aid
    'SH.MED.PHYS.ZS': 'physicians_per_1000',
    
    # Hospital beds per 1,000 people — physical infrastructure
    # Higher = more facilities to absorb health funding
    'SH.MED.BEDS.ZS': 'hospital_beds_per_1000',
    
    # Government debt (% of GDP) — fiscal space
    # Higher debt = less ability to co-fund or sustain programs after aid ends
    'GC.DOD.TOTL.GD.ZS': 'govt_debt_pct_gdp',
    
    # Under-5 mortality rate (per 1,000 live births) — health outcome
    # Higher = worse health outcomes = more need for health funding
    'SH.DYN.MORT': 'under5_mortality',
    
    # Life expectancy at birth — overall health outcome
    # Lower = worse baseline health
    'SP.DYN.LE00.IN': 'life_expectancy',
    
    # Immunization, measles (% of children ages 12-23 months) — system delivery capacity  
    # Higher = health system can actually reach people
    'SH.IMM.MEAS': 'measles_vaccination_pct',
    
    # Out-of-pocket health expenditure (% of current health expenditure)
    # Higher = people pay more themselves = system is failing them
    'SH.XPD.OOPC.CH.ZS': 'out_of_pocket_pct',
    
    # Domestic general government health expenditure (% of GDP)
    # Shows government commitment to health specifically
    'SH.XPD.GHED.GD.ZS': 'govt_health_expenditure_pct_gdp'
}

# ============================================================
# FETCH DATA — most recent available (2018-2024 window)
# ============================================================
wb_data = {}
fetch_log = []

for indicator_code, indicator_name in indicators.items():
    url = f"https://api.worldbank.org/v2/country/{country_str}/indicator/{indicator_code}?format=json&date=2018:2024&per_page=1000"
    try:
        resp = requests.get(url, timeout=20)
        data = resp.json()
        
        countries_found = 0
        if len(data) > 1 and data[1]:
            for entry in data[1]:
                if entry['value'] is not None:
                    iso3 = entry['countryiso3code']
                    year = int(entry['date'])
                    if iso3 not in wb_data:
                        wb_data[iso3] = {}
                    # Keep most recent value per country
                    if indicator_name not in wb_data[iso3] or year > wb_data[iso3].get(f'{indicator_name}_year', 0):
                        wb_data[iso3][indicator_name] = entry['value']
                        wb_data[iso3][f'{indicator_name}_year'] = year
            
            countries_found = len([k for k, v in wb_data.items() if indicator_name in v])
        
        status = f"✓ {indicator_name}: {countries_found}/18 countries"
        print(status)
        fetch_log.append(status)
        
    except Exception as e:
        status = f"✗ {indicator_name}: FAILED — {e}"
        print(status)
        fetch_log.append(status)

# ============================================================
# BUILD WORLD BANK DATAFRAME
# ============================================================
wb_df = pd.DataFrame.from_dict(wb_data, orient='index')
wb_df.index.name = 'Country ISO3'
wb_df = wb_df.reset_index()

# Drop the year tracking columns
year_cols = [c for c in wb_df.columns if c.endswith('_year')]
wb_df_display = wb_df.drop(columns=year_cols, errors='ignore')

print(f"\n{'='*100}")
print(f"WORLD BANK DATA — {len(wb_df)} countries retrieved")
print(f"{'='*100}")
print(wb_df_display.to_string())

# ============================================================
# COMPUTE ABSORPTION CAPACITY INDEX
# ============================================================

# Positive indicators (higher = better absorption capacity)
positive_indicators = ['gdp_per_capita', 'health_expenditure_pct_gdp', 'physicians_per_1000', 
                        'hospital_beds_per_1000', 'life_expectancy', 'measles_vaccination_pct',
                        'govt_health_expenditure_pct_gdp']

# Negative indicators (higher = WORSE absorption capacity)
negative_indicators = ['govt_debt_pct_gdp', 'under5_mortality', 'out_of_pocket_pct']

# Normalize positive indicators to 0-1
for col in positive_indicators:
    if col in wb_df.columns:
        min_v = wb_df[col].min()
        max_v = wb_df[col].max()
        if max_v > min_v:
            wb_df[f'{col}_norm'] = (wb_df[col] - min_v) / (max_v - min_v)
        else:
            wb_df[f'{col}_norm'] = 0.5

# Normalize negative indicators (invert: higher raw = lower normalized)
for col in negative_indicators:
    if col in wb_df.columns:
        min_v = wb_df[col].min()
        max_v = wb_df[col].max()
        if max_v > min_v:
            wb_df[f'{col}_norm'] = 1 - ((wb_df[col] - min_v) / (max_v - min_v))
        else:
            wb_df[f'{col}_norm'] = 0.5

# Composite absorption capacity (mean of all available normalized indicators)
norm_cols = [c for c in wb_df.columns if c.endswith('_norm')]
wb_df['absorption_capacity'] = wb_df[norm_cols].mean(axis=1)

print(f"\n{'='*100}")
print(f"ABSORPTION CAPACITY INDEX (0 = no capacity, 1 = full capacity)")
print(f"{'='*100}")
print(f"Components used: {norm_cols}")
print(wb_df[['Country ISO3', 'gdp_per_capita', 'physicians_per_1000', 'hospital_beds_per_1000',
             'life_expectancy', 'under5_mortality', 'measles_vaccination_pct', 
             'absorption_capacity']].sort_values('absorption_capacity').to_string())

# ============================================================
# MERGE WITH CRISIS COMPASS MASTER TABLE
# ============================================================

# Load the scored master if not in memory
try:
    _ = scored.shape
except:
    scored = pd.read_csv(f"{base}crisis_compass_master.csv")
    # Recompute norms
    scored['health_in_need_24'] = pd.to_numeric(scored['health_in_need_24'], errors='coerce')
    scored['health_coverage'] = pd.to_numeric(scored['health_coverage'], errors='coerce')
    scored['health_funding_per_person_in_need'] = pd.to_numeric(scored['health_funding_per_person_in_need'], errors='coerce')

merge_cols = ['Country ISO3', 'absorption_capacity', 'gdp_per_capita', 'physicians_per_1000',
              'hospital_beds_per_1000', 'life_expectancy', 'under5_mortality', 
              'measles_vaccination_pct', 'out_of_pocket_pct', 'health_expenditure_pct_gdp',
              'govt_health_expenditure_pct_gdp', 'govt_debt_pct_gdp']
merge_cols = [c for c in merge_cols if c in wb_df.columns]

scored = scored.merge(wb_df[merge_cols], on='Country ISO3', how='left')

# ============================================================
# UPDATED INVISIBLE CRISIS SCORE V2 (with Absorption)
# ============================================================

# Re-normalize components for scored countries with valid data
scored_valid = scored[
    (scored['health_required'] > 0) & 
    (scored['health_coverage'] != np.inf) &
    (scored['health_in_need_24'].notna()) &
    (scored['absorption_capacity'].notna())
].copy()

for col in ['health_in_need_24', 'health_gap']:
    min_v = scored_valid[col].min()
    max_v = scored_valid[col].max()
    scored_valid[f'{col}_norm'] = (scored_valid[col] - min_v) / (max_v - min_v) if max_v > min_v else 0

scored_valid['inverse_coverage_norm'] = 1 - (scored_valid['health_coverage'] / scored_valid['health_coverage'].max())

inv_per_cap = 1 / scored_valid['health_funding_per_person_in_need'].replace(0, np.nan)
scored_valid['inverse_per_capita_norm'] = (inv_per_cap - inv_per_cap.min()) / (inv_per_cap.max() - inv_per_cap.min())

# Absorption: lower = worse = higher score component
scored_valid['inverse_absorption_norm'] = 1 - (scored_valid['absorption_capacity'] / scored_valid['absorption_capacity'].max())

# V2 Score: 4 components equally weighted
scored_valid['invisible_crisis_score_v2'] = (
    scored_valid['health_in_need_24_norm'] * 0.25 +
    scored_valid['inverse_coverage_norm'] * 0.25 +
    scored_valid['inverse_per_capita_norm'] * 0.25 +
    scored_valid['inverse_absorption_norm'] * 0.25
)

scored_valid['crisis_rank_v2'] = scored_valid['invisible_crisis_score_v2'].rank(ascending=False).astype(int)

print(f"\n{'='*100}")
print(f"INVISIBLE CRISIS SCORE V2 — With Absorption Capacity")
print(f"{'='*100}")
print(f"Formula: 25% scale of need + 25% underfunding + 25% low per-capita + 25% low absorption")
print(scored_valid[['Country ISO3', 'crisis_rank_v2', 'invisible_crisis_score_v2',
                     'health_in_need_24', 'health_coverage', 'health_funding_per_person_in_need',
                     'absorption_capacity', 'gdp_per_capita', 'under5_mortality',
                     'physicians_per_1000']].sort_values('crisis_rank_v2').to_string())

# ============================================================
# KEY NEW INSIGHT: Absorption vs Coverage
# ============================================================
print(f"\n{'='*100}")
print(f"NEW INSIGHT: High Absorption + Low Coverage = Wasted Potential")
print(f"{'='*100}")
print("Countries with above-median absorption but below-median health coverage:")
med_abs = scored_valid['absorption_capacity'].median()
med_cov = scored_valid['health_coverage'].median()
wasted = scored_valid[(scored_valid['absorption_capacity'] > med_abs) & (scored_valid['health_coverage'] < med_cov)]
if len(wasted) > 0:
    print(wasted[['Country ISO3', 'absorption_capacity', 'health_coverage', 'health_gap']].to_string())
else:
    print("  (No countries in this quadrant — most low-coverage countries also have low absorption)")

print("\nCountries with low absorption + high coverage = funding hitting a wall:")
wall = scored_valid[(scored_valid['absorption_capacity'] < med_abs) & (scored_valid['health_coverage'] > med_cov)]
if len(wall) > 0:
    print(wall[['Country ISO3', 'absorption_capacity', 'health_coverage', 'health_gap']].to_string())

# ============================================================
# SAVE EVERYTHING
# ============================================================
# Full master with World Bank data
scored_valid.to_csv(f"{base}crisis_compass_master_v2.csv", index=False)

# Figma export v2
export_cols = ['Country ISO3', 'crisis_rank_v2', 'invisible_crisis_score_v2',
               'health_in_need_24', 'health_in_need_25', 'health_need_change_pct',
               'health_required', 'health_funded', 'health_gap', 'health_coverage',
               'food_in_need_24', 'food_required', 'food_funded', 'food_coverage',
               'health_funding_per_person_in_need', 'health_vs_food_coverage_gap',
               'absorption_capacity', 'gdp_per_capita', 'physicians_per_1000',
               'under5_mortality', 'life_expectancy', 'measles_vaccination_pct',
               'total_pop', 'food_to_health_lag']
export_cols = [c for c in export_cols if c in scored_valid.columns]

figma_v2 = scored_valid[export_cols].copy()
for col in figma_v2.select_dtypes(include=[np.number]).columns:
    figma_v2[col] = figma_v2[col].round(4)
figma_v2 = figma_v2.sort_values('crisis_rank_v2')
figma_v2.to_csv(f"{base}figma_export_v2.csv", index=False)

# World Bank standalone
wb_df.to_csv(f"{base}world_bank_indicators.csv", index=False)

print(f"\n{'='*100}")
print(f"FILES SAVED")
print(f"{'='*100}")
print(f"  {base}crisis_compass_master_v2.csv — full analysis with World Bank ({len(scored_valid)} countries)")
print(f"  {base}figma_export_v2.csv — for Figma Make")
print(f"  {base}world_bank_indicators.csv — standalone World Bank data")
print(f"\nCSV for Figma Make:")
print(figma_v2.to_csv(index=False))

In [0]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

fig = plt.figure(figsize=(26, 32))
fig.suptitle('CRISIS COMPASS: Humanitarian Health Funding × Economic Absorption Capacity', 
             fontsize=20, fontweight='bold', y=0.98)

# ============================================================
# PLOT 1: THE QUADRANT — Absorption vs Coverage (KEY SLIDE)
# ============================================================
ax1 = fig.add_subplot(3, 2, 1)
sv = scored_valid.copy()

sizes = sv['health_in_need_24'] / sv['health_in_need_24'].max() * 800 + 80

scatter = ax1.scatter(sv['absorption_capacity'], sv['health_coverage'] * 100, 
                       s=sizes, c=sv['invisible_crisis_score_v2'], cmap='RdYlGn_r', 
                       edgecolors='black', linewidth=0.7, alpha=0.85, zorder=5)

for _, row in sv.iterrows():
    ax1.annotate(row['Country ISO3'], 
                (row['absorption_capacity'], row['health_coverage']*100),
                fontsize=8, fontweight='bold', ha='center', va='bottom')

# Quadrant lines at medians
med_abs = sv['absorption_capacity'].median()
med_cov = sv['health_coverage'].median() * 100
ax1.axhline(med_cov, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax1.axvline(med_abs, color='gray', linewidth=1, linestyle='--', alpha=0.5)

# Quadrant labels
ax1.text(0.05, 22, 'MOST INVISIBLE\nLow capacity + Low funding', fontsize=8, 
         color='#c0392b', fontweight='bold', style='italic')
ax1.text(0.55, 22, 'WASTED POTENTIAL\nHigh capacity + Low funding', fontsize=8, 
         color='#e67e22', fontweight='bold', style='italic')
ax1.text(0.05, 2, 'FUNDING HITTING A WALL\nLow capacity + Higher funding', fontsize=8, 
         color='#8e44ad', fontweight='bold', style='italic')
ax1.text(0.55, 2, 'EFFECTIVE\nHigh capacity + Higher funding', fontsize=8, 
         color='#27ae60', fontweight='bold', style='italic')

ax1.set_xlabel('Absorption Capacity Index (World Bank)', fontsize=11)
ax1.set_ylabel('Health Funding Coverage (%)', fontsize=11)
ax1.set_title('The Absorption–Coverage Quadrant\n(bubble size = people in need, color = crisis score)', fontsize=12)
plt.colorbar(scatter, ax=ax1, label='Crisis Score', shrink=0.8)

# ============================================================
# PLOT 2: Invisible Crisis Score V2 — Ranked with components
# ============================================================
ax2 = fig.add_subplot(3, 2, 2)
ranked = sv.sort_values('invisible_crisis_score_v2', ascending=True)

colors_bar = plt.cm.RdYlGn_r(ranked['invisible_crisis_score_v2'] / ranked['invisible_crisis_score_v2'].max())
bars = ax2.barh(range(len(ranked)), ranked['invisible_crisis_score_v2'], color=colors_bar, 
                edgecolor='black', linewidth=0.3)
ax2.set_yticks(range(len(ranked)))
ax2.set_yticklabels(ranked['Country ISO3'], fontsize=9)
ax2.set_xlabel('Invisible Crisis Score V2', fontsize=11)
ax2.set_title('Invisible Crisis Score V2 Rankings\n(Now includes Absorption Capacity)', fontsize=12)

for i, (_, row) in enumerate(ranked.iterrows()):
    ax2.text(row['invisible_crisis_score_v2'] + 0.01, i, 
            f"{row['invisible_crisis_score_v2']:.2f} (abs: {row['absorption_capacity']:.2f})", 
            va='center', fontsize=7)

# ============================================================
# PLOT 3: Absorption Capacity Breakdown — What makes a country capable?
# ============================================================
ax3 = fig.add_subplot(3, 2, 3)
breakdown_cols = ['gdp_per_capita', 'physicians_per_1000', 'under5_mortality', 
                   'measles_vaccination_pct', 'life_expectancy']
breakdown = sv[['Country ISO3'] + breakdown_cols + ['absorption_capacity']].sort_values('absorption_capacity')

# Horizontal bar of absorption capacity, colored by under-5 mortality
colors_abs = plt.cm.RdYlGn_r(breakdown['under5_mortality'] / breakdown['under5_mortality'].max())
ax3.barh(range(len(breakdown)), breakdown['absorption_capacity'], color=colors_abs, 
         edgecolor='black', linewidth=0.3)
ax3.set_yticks(range(len(breakdown)))
ax3.set_yticklabels(breakdown['Country ISO3'], fontsize=9)
ax3.set_xlabel('Absorption Capacity Index', fontsize=11)
ax3.set_title('Absorption Capacity by Country\n(Color = under-5 mortality: red=high, green=low)', fontsize=12)

for i, (_, row) in enumerate(breakdown.iterrows()):
    label = f"GDP ${row['gdp_per_capita']:,.0f} | {row['physicians_per_1000']:.1f} docs/1k" if pd.notna(row['gdp_per_capita']) else ""
    ax3.text(row['absorption_capacity'] + 0.01, i, label, va='center', fontsize=6.5)

# ============================================================
# PLOT 4: Scatter — GDP per capita vs Health Funding per Person
# ============================================================
ax4 = fig.add_subplot(3, 2, 4)
gdp_valid = sv.dropna(subset=['gdp_per_capita'])

ax4.scatter(gdp_valid['gdp_per_capita'], gdp_valid['health_funding_per_person_in_need'],
           s=200, c=gdp_valid['invisible_crisis_score_v2'], cmap='RdYlGn_r', 
           edgecolors='black', linewidth=0.7, alpha=0.85)

for _, row in gdp_valid.iterrows():
    ax4.annotate(row['Country ISO3'], 
                (row['gdp_per_capita'], row['health_funding_per_person_in_need']),
                fontsize=8, fontweight='bold', ha='center', va='bottom')

ax4.set_xlabel('GDP per Capita ($)', fontsize=11)
ax4.set_ylabel('Health Funding per Person in Need ($)', fontsize=11)
ax4.set_title('Economic Strength vs Health Funding Received\n(Richer countries don\'t necessarily get more health aid)', fontsize=12)

# ============================================================
# PLOT 5: Under-5 Mortality vs Health Coverage
# ============================================================
ax5 = fig.add_subplot(3, 2, 5)
ax5.scatter(sv['health_coverage'] * 100, sv['under5_mortality'],
           s=sv['health_in_need_24'] / sv['health_in_need_24'].max() * 600 + 80,
           c=sv['absorption_capacity'], cmap='RdYlGn', 
           edgecolors='black', linewidth=0.7, alpha=0.85)

for _, row in sv.iterrows():
    ax5.annotate(row['Country ISO3'], 
                (row['health_coverage']*100, row['under5_mortality']),
                fontsize=8, fontweight='bold', ha='center', va='bottom')

ax5.set_xlabel('Health Funding Coverage (%)', fontsize=11)
ax5.set_ylabel('Under-5 Mortality (per 1,000 births)', fontsize=11)
ax5.set_title('Does More Health Funding → Lower Child Mortality?\n(Color = absorption capacity: green=high)', fontsize=12)

# Add trend annotation
ax5.text(20, 10, '← More funding doesn\'t always help\n     without absorption capacity', 
         fontsize=8, color='gray', style='italic')

# ============================================================
# PLOT 6: Food→Health Lag + Absorption (the full story)
# ============================================================
ax6 = fig.add_subplot(3, 2, 6)
temporal = sv.dropna(subset=['health_need_change_pct', 'food_need_change_pct']).copy()

sizes_t = (1 - temporal['absorption_capacity']) * 500 + 50  # Bigger = lower absorption
colors_t = ['#e74c3c' if row['food_to_health_lag'] else '#3498db' for _, row in temporal.iterrows()]

ax6.scatter(temporal['food_need_change_pct'], temporal['health_need_change_pct'],
           s=sizes_t, c=colors_t, edgecolors='black', linewidth=0.7, alpha=0.8, zorder=5)

for _, row in temporal.iterrows():
    ax6.annotate(f"{row['Country ISO3']}\n(abs:{row['absorption_capacity']:.2f})", 
                (row['food_need_change_pct'], row['health_need_change_pct']),
                fontsize=7, fontweight='bold', ha='center', va='bottom')

ax6.axhline(0, color='gray', linewidth=0.8)
ax6.axvline(0, color='gray', linewidth=0.8)
ax6.fill_between([-80, 0], [0, 0], [300, 300], alpha=0.08, color='red')
ax6.text(-45, 200, 'FOOD→HEALTH LAG\n+ Low Absorption = Perfect Storm', 
         fontsize=9, color='red', ha='center', style='italic', fontweight='bold')

ax6.set_xlabel('Food Security Need Change 2024→2025 (%)', fontsize=11)
ax6.set_ylabel('Health Need Change 2024→2025 (%)', fontsize=11)
ax6.set_title('Food→Health Lag Signal + Absorption Capacity\n(Bubble size = inverse absorption: bigger = less capacity)', fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(f"{base}crisis_compass_v2_visualizations.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved to {base}crisis_compass_v2_visualizations.png")

In [0]:
# ============================================================
# CRISIS COMPASS — COMPLETE PIPELINE (Single Cell)
# ============================================================
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

base = "/Volumes/workspace/default/foodtohealth/"

# ============================================================
# PART 1: LOAD ALL 17 DATASETS
# ============================================================
print("PART 1: Loading all datasets...")

hno_24 = pd.read_csv(f"{base}hpc_hno_2024.csv", low_memory=False)
hno_25 = pd.read_csv(f"{base}hpc_hno_2025.csv", low_memory=False)
hno_26 = pd.read_csv(f"{base}hpc_hno_2026.csv", low_memory=False)
fund_cluster = pd.read_csv(f"{base}fts_requirements_funding_cluster_global.csv")
fund_global = pd.read_csv(f"{base}fts_requirements_funding_global.csv")
fund_globalcluster = pd.read_csv(f"{base}fts_requirements_funding_globalcluster_global.csv")
fund_covid = pd.read_csv(f"{base}fts_requirements_funding_covid_global.csv")
incoming = pd.read_csv(f"{base}fts_incoming_funding_global.csv")
outgoing = pd.read_csv(f"{base}fts_outgoing_funding_global.csv")
pop_0 = pd.read_csv(f"{base}cod_population_admin0.csv")
pop_1 = pd.read_csv(f"{base}cod_population_admin1.csv")
pop_4 = pd.read_csv(f"{base}cod_population_admin4.csv")
hrp = pd.read_csv(f"{base}humanitarian-response-plans.csv")
allocations = pd.read_csv(f"{base}Allocations__20260221_193441_UTC.csv")
contributions = pd.read_csv(f"{base}Contributions__20260221_193441_UTC.csv")
metadata_hno = pd.read_csv(f"{base}metadata-global-hpc-hno-2025.csv")

# Remove HXL tag rows (first row starting with '#')
for df in [hno_24, hno_25, fund_globalcluster, fund_global, fund_cluster, 
           fund_covid, incoming, outgoing, hrp]:
    if str(df.iloc[0, 0]).startswith('#'):
        df.drop(0, inplace=True)
        df.reset_index(drop=True, inplace=True)

# Fix numeric types
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    for df in [hno_24, hno_25, hno_26]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'percentFunded']:
    for df in [fund_globalcluster, fund_global, fund_cluster]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'covidFunding']:
    if col in fund_covid.columns:
        fund_covid[col] = pd.to_numeric(fund_covid[col], errors='coerce')

incoming['amountUSD'] = pd.to_numeric(incoming['amountUSD'], errors='coerce')
outgoing['amountUSD'] = pd.to_numeric(outgoing['amountUSD'], errors='coerce')
pop_0['Population'] = pd.to_numeric(pop_0['Population'], errors='coerce')
pop_1['Population'] = pd.to_numeric(pop_1['Population'], errors='coerce')
for col in ['origRequirements', 'revisedRequirements']:
    hrp[col] = pd.to_numeric(hrp[col], errors='coerce')
allocations['Budget'] = pd.to_numeric(allocations['Budget'], errors='coerce')
for col in ['Paid', 'Pledged', 'Total']:
    contributions[col] = pd.to_numeric(contributions[col], errors='coerce')

print(f"  Loaded: HNO 2024 ({len(hno_24)} rows), HNO 2025 ({len(hno_25)}), HNO 2026 ({len(hno_26)})")
print(f"  Funding: cluster ({len(fund_cluster)}), global ({len(fund_global)}), globalcluster ({len(fund_globalcluster)}), covid ({len(fund_covid)})")
print(f"  Flows: incoming ({len(incoming)}), outgoing ({len(outgoing)})")
print(f"  Population: L0 ({len(pop_0)}), L1 ({len(pop_1)}), L4 ({len(pop_4)})")
print(f"  Other: HRP ({len(hrp)}), allocations ({len(allocations)}), contributions ({len(contributions)})")

# ============================================================
# PART 2: BUILD NEEDS TABLES FROM HNO
# ============================================================
print("\nPART 2: Building needs tables...")

health_24 = hno_24[hno_24['Cluster'] == 'HEA'].groupby('Country ISO3').agg(
    health_in_need_24=('In Need', 'sum'), health_targeted_24=('Targeted', 'sum'),
    health_affected_24=('Affected', 'sum'), health_reached_24=('Reached', 'sum')
).reset_index()

health_25 = hno_25[hno_25['Cluster'] == 'HEA'].groupby('Country ISO3').agg(
    health_in_need_25=('In Need', 'sum'), health_targeted_25=('Targeted', 'sum')
).reset_index()

food_24 = hno_24[hno_24['Cluster'] == 'FSC'].groupby('Country ISO3').agg(
    food_in_need_24=('In Need', 'sum')).reset_index()

food_25 = hno_25[hno_25['Cluster'] == 'FSC'].groupby('Country ISO3').agg(
    food_in_need_25=('In Need', 'sum')).reset_index()

all_needs_24 = hno_24.groupby('Country ISO3').agg(
    total_in_need_24=('In Need', 'sum'), total_population_24=('Population', 'sum')
).reset_index()

# All clusters for each year
all_clusters_24 = hno_24.groupby(['Country ISO3', 'Cluster']).agg(
    in_need=('In Need', 'sum')).reset_index()

print(f"  Health needs: {len(health_24)} countries (2024), {len(health_25)} (2025)")
print(f"  Food needs: {len(food_24)} countries (2024), {len(food_25)} (2025)")

# ============================================================
# PART 3: BUILD FUNDING TABLES
# ============================================================
print("\nPART 3: Building funding tables...")

# Health funding (from fund_globalcluster — cleanest cluster names)
health_funding = fund_globalcluster[
    fund_globalcluster['cluster'].str.contains('^Health$', na=False, regex=True)
].groupby(['countryCode', 'year']).agg(
    health_required=('requirements', 'sum'), health_funded=('funding', 'sum')
).reset_index()

food_funding = fund_globalcluster[
    fund_globalcluster['cluster'] == 'Food Security'
].groupby(['countryCode', 'year']).agg(
    food_required=('requirements', 'sum'), food_funded=('funding', 'sum')
).reset_index()

total_funding = fund_global.groupby(['countryCode', 'year']).agg(
    total_required=('requirements', 'sum'), total_funded=('funding', 'sum')
).reset_index()

# Latest year per country
health_funding_latest = health_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')
food_funding_latest = food_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')
total_funding_latest = total_funding.sort_values('year', ascending=False).drop_duplicates('countryCode')

# COVID funding
covid_funding = fund_covid.groupby('countryCode').agg(
    covid_total_funding=('funding', 'sum'), covid_specific_funding=('covidFunding', 'sum')
).reset_index()

# Incoming health donor flows
health_incoming = incoming[incoming['destGlobalClusters'].str.contains('Health', na=False, case=False)]
health_donor_summary = health_incoming.groupby('srcOrganization')['amountUSD'].sum().sort_values(ascending=False)

# Pooled fund allocations
alloc_by_country = allocations.groupby('PooledFund').agg(
    total_alloc=('Budget', 'sum'), n_allocations=('Budget', 'count')
).sort_values('total_alloc', ascending=False).reset_index()

# Population
total_pop = pop_0.groupby('ISO3')['Population'].sum().reset_index().rename(columns={'Population': 'total_pop'})

print(f"  Health funding: {health_funding_latest['countryCode'].nunique()} countries")
print(f"  Food funding: {food_funding_latest['countryCode'].nunique()} countries")
print(f"  Pooled fund total: ${allocations['Budget'].sum():,.0f}")

# ============================================================
# PART 4: MASTER JOIN
# ============================================================
print("\nPART 4: Building master table...")

master = health_24.merge(health_25, on='Country ISO3', how='outer')
master = master.merge(food_24, on='Country ISO3', how='outer')
master = master.merge(food_25, on='Country ISO3', how='outer')
master = master.merge(all_needs_24, on='Country ISO3', how='outer')
master = master.merge(health_funding_latest[['countryCode', 'health_required', 'health_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left')
master = master.merge(food_funding_latest[['countryCode', 'food_required', 'food_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_food'))
master = master.merge(total_funding_latest[['countryCode', 'total_required', 'total_funded']], 
                       left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_total'))
master = master.merge(total_pop, left_on='Country ISO3', right_on='ISO3', how='left')
master = master.merge(covid_funding, left_on='Country ISO3', right_on='countryCode', how='left', suffixes=('', '_covid'))
master = master.drop(columns=[c for c in master.columns if c.startswith('countryCode') or c == 'ISO3'], errors='ignore')

# Compute metrics
master['health_gap'] = master['health_required'] - master['health_funded']
master['food_gap'] = master['food_required'] - master['food_funded']
master['total_gap'] = master['total_required'] - master['total_funded']
master['health_coverage'] = master['health_funded'] / master['health_required']
master['food_coverage'] = master['food_funded'] / master['food_required']
master['total_coverage'] = master['total_funded'] / master['total_required']
master['health_funding_per_person_in_need'] = master['health_funded'] / master['health_in_need_24']
master['health_gap_per_person_in_need'] = master['health_gap'] / master['health_in_need_24']
master['total_funding_per_capita'] = master['total_funded'] / master['total_pop']
master['health_share_of_funding'] = master['health_funded'] / master['total_funded']
master['health_share_of_requirements'] = master['health_required'] / master['total_required']
master['health_vs_food_coverage_gap'] = master['food_coverage'] - master['health_coverage']
master['health_need_change_pct'] = (master['health_in_need_25'] - master['health_in_need_24']) / master['health_in_need_24'] * 100
master['food_need_change_pct'] = (master['food_in_need_25'] - master['food_in_need_24']) / master['food_in_need_24'] * 100
master['food_to_health_lag'] = (
    (master['food_in_need_24'] > master['food_in_need_25']) &
    (master['health_in_need_25'] > master['health_in_need_24'])
)

print(f"  Master table: {len(master)} countries, {len(master.columns)} columns")

# ============================================================
# PART 5: WORLD BANK API
# ============================================================
print("\nPART 5: Fetching World Bank data...")

countries_list = master['Country ISO3'].dropna().tolist()
country_str = ';'.join(countries_list)

indicators = {
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'SH.XPD.CHEX.GD.ZS': 'health_expenditure_pct_gdp',
    'SH.MED.PHYS.ZS': 'physicians_per_1000',
    'SH.MED.BEDS.ZS': 'hospital_beds_per_1000',
    'GC.DOD.TOTL.GD.ZS': 'govt_debt_pct_gdp',
    'SH.DYN.MORT': 'under5_mortality',
    'SP.DYN.LE00.IN': 'life_expectancy',
    'SH.IMM.MEAS': 'measles_vaccination_pct',
    'SH.XPD.OOPC.CH.ZS': 'out_of_pocket_pct',
    'SH.XPD.GHED.GD.ZS': 'govt_health_expenditure_pct_gdp'
}

wb_data = {}
for indicator_code, indicator_name in indicators.items():
    url = f"https://api.worldbank.org/v2/country/{country_str}/indicator/{indicator_code}?format=json&date=2018:2024&per_page=1000"
    try:
        resp = requests.get(url, timeout=20)
        data = resp.json()
        if len(data) > 1 and data[1]:
            for entry in data[1]:
                if entry['value'] is not None:
                    iso3 = entry['countryiso3code']
                    year = int(entry['date'])
                    if iso3 not in wb_data:
                        wb_data[iso3] = {}
                    if indicator_name not in wb_data[iso3] or year > wb_data[iso3].get(f'{indicator_name}_year', 0):
                        wb_data[iso3][indicator_name] = entry['value']
                        wb_data[iso3][f'{indicator_name}_year'] = year
        found = len([k for k, v in wb_data.items() if indicator_name in v])
        print(f"  ✓ {indicator_name}: {found} countries")
    except Exception as e:
        print(f"  ✗ {indicator_name}: {e}")

wb_df = pd.DataFrame.from_dict(wb_data, orient='index')
wb_df.index.name = 'Country ISO3'
wb_df = wb_df.reset_index()
year_cols = [c for c in wb_df.columns if c.endswith('_year')]
wb_df = wb_df.drop(columns=year_cols, errors='ignore')

# Compute Absorption Capacity
positive = ['gdp_per_capita', 'health_expenditure_pct_gdp', 'physicians_per_1000', 
            'hospital_beds_per_1000', 'life_expectancy', 'measles_vaccination_pct', 'govt_health_expenditure_pct_gdp']
negative = ['govt_debt_pct_gdp', 'under5_mortality', 'out_of_pocket_pct']

for col in positive:
    if col in wb_df.columns:
        mn, mx = wb_df[col].min(), wb_df[col].max()
        wb_df[f'{col}_norm'] = (wb_df[col] - mn) / (mx - mn) if mx > mn else 0.5

for col in negative:
    if col in wb_df.columns:
        mn, mx = wb_df[col].min(), wb_df[col].max()
        wb_df[f'{col}_norm'] = 1 - ((wb_df[col] - mn) / (mx - mn)) if mx > mn else 0.5

norm_cols = [c for c in wb_df.columns if c.endswith('_norm')]
wb_df['absorption_capacity'] = wb_df[norm_cols].mean(axis=1)

# Merge World Bank into master
merge_cols = ['Country ISO3', 'absorption_capacity', 'gdp_per_capita', 'physicians_per_1000',
              'hospital_beds_per_1000', 'life_expectancy', 'under5_mortality', 
              'measles_vaccination_pct', 'out_of_pocket_pct', 'health_expenditure_pct_gdp',
              'govt_health_expenditure_pct_gdp', 'govt_debt_pct_gdp']
merge_cols = [c for c in merge_cols if c in wb_df.columns]
master = master.merge(wb_df[merge_cols], on='Country ISO3', how='left')

print(f"  World Bank data merged for {wb_df['Country ISO3'].nunique()} countries")

# ============================================================
# PART 6: INVISIBLE CRISIS SCORE V2
# ============================================================
print("\nPART 6: Computing Invisible Crisis Score V2...")

scored = master[
    (master['health_required'] > 0) & 
    (master['health_coverage'] != np.inf) &
    (master['health_in_need_24'].notna()) &
    (master['absorption_capacity'].notna())
].copy()

# Normalize components
for col in ['health_in_need_24', 'health_gap']:
    mn, mx = scored[col].min(), scored[col].max()
    scored[f'{col}_norm'] = (scored[col] - mn) / (mx - mn) if mx > mn else 0

scored['inverse_coverage_norm'] = 1 - (scored['health_coverage'] / scored['health_coverage'].max())

inv_pc = 1 / scored['health_funding_per_person_in_need'].replace(0, np.nan)
scored['inverse_per_capita_norm'] = (inv_pc - inv_pc.min()) / (inv_pc.max() - inv_pc.min())

scored['inverse_absorption_norm'] = 1 - (scored['absorption_capacity'] / scored['absorption_capacity'].max())

scored['invisible_crisis_score_v2'] = (
    scored['health_in_need_24_norm'] * 0.25 +
    scored['inverse_coverage_norm'] * 0.25 +
    scored['inverse_per_capita_norm'] * 0.25 +
    scored['inverse_absorption_norm'] * 0.25
)
scored['crisis_rank_v2'] = scored['invisible_crisis_score_v2'].rank(ascending=False).astype(int)

# Also compute V1 (without absorption) for comparison
scored['invisible_crisis_score_v1'] = (
    scored['health_in_need_24_norm'] * 0.35 +
    scored['inverse_coverage_norm'] * 0.35 +
    scored['inverse_per_capita_norm'] * 0.30
)
scored['crisis_rank_v1'] = scored['invisible_crisis_score_v1'].rank(ascending=False).astype(int)

# Health need share
scored['health_need_share'] = scored['health_in_need_24'] / scored['total_in_need_24']

print(f"  Scored {len(scored)} countries")

# ============================================================
# PART 7: SENSITIVITY ANALYSIS
# ============================================================
print("\nPART 7: Sensitivity analysis...")

schemes = {
    'Equal (25/25/25/25)': (0.25, 0.25, 0.25, 0.25),
    'Need-heavy (40/20/20/20)': (0.40, 0.20, 0.20, 0.20),
    'Coverage-heavy (20/40/20/20)': (0.20, 0.40, 0.20, 0.20),
    'Absorption-heavy (20/20/20/40)': (0.20, 0.20, 0.20, 0.40)
}

rankings = {}
for name, (w1, w2, w3, w4) in schemes.items():
    scored[f'score_{name}'] = (
        scored['health_in_need_24_norm'] * w1 +
        scored['inverse_coverage_norm'] * w2 +
        scored['inverse_per_capita_norm'] * w3 +
        scored['inverse_absorption_norm'] * w4
    )
    rankings[name] = scored.sort_values(f'score_{name}', ascending=False)['Country ISO3'].tolist()

from collections import Counter
all_top5 = []
for name in schemes:
    all_top5.extend(rankings[name][:5])
stable_top5 = [c for c, count in Counter(all_top5).items() if count == len(schemes)]

for name in schemes:
    print(f"  {name}: {rankings[name][:8]}")
print(f"  STABLE TOP 5: {stable_top5}")

# ============================================================
# PART 8: ALL VISUALIZATIONS (8 plots)
# ============================================================
print("\nPART 8: Generating visualizations...")

fig = plt.figure(figsize=(28, 38))
fig.suptitle('CRISIS COMPASS: Humanitarian Health Funding × Economic Absorption Capacity\nHacklytics 2026 — 7 UN Datasets + 10 World Bank Indicators → 5 New Metrics', 
             fontsize=18, fontweight='bold', y=0.99)

# --- PLOT 1: Absorption–Coverage Quadrant ---
ax1 = fig.add_subplot(4, 2, 1)
sv = scored.copy()
sizes = sv['health_in_need_24'] / sv['health_in_need_24'].max() * 800 + 80
scatter = ax1.scatter(sv['absorption_capacity'], sv['health_coverage'] * 100, 
                       s=sizes, c=sv['invisible_crisis_score_v2'], cmap='RdYlGn_r', 
                       edgecolors='black', linewidth=0.7, alpha=0.85, zorder=5)
for _, row in sv.iterrows():
    ax1.annotate(row['Country ISO3'], (row['absorption_capacity'], row['health_coverage']*100),
                fontsize=8, fontweight='bold', ha='center', va='bottom')
med_abs = sv['absorption_capacity'].median()
med_cov = sv['health_coverage'].median() * 100
ax1.axhline(med_cov, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax1.axvline(med_abs, color='gray', linewidth=1, linestyle='--', alpha=0.5)
ax1.text(0.05, max(sv['health_coverage']*100)*0.85, 'MOST INVISIBLE\nLow capacity + Low funding', fontsize=7, color='#c0392b', fontweight='bold')
ax1.text(0.55, max(sv['health_coverage']*100)*0.85, 'WASTED POTENTIAL\nHigh capacity + Low funding', fontsize=7, color='#e67e22', fontweight='bold')
ax1.text(0.05, 1, 'WALL\nLow capacity + More funding', fontsize=7, color='#8e44ad', fontweight='bold')
ax1.text(0.55, 1, 'EFFECTIVE\nHigh capacity + More funding', fontsize=7, color='#27ae60', fontweight='bold')
ax1.set_xlabel('Absorption Capacity (World Bank)', fontsize=10)
ax1.set_ylabel('Health Coverage (%)', fontsize=10)
ax1.set_title('The Absorption–Coverage Quadrant', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax1, label='Crisis Score V2', shrink=0.8)

# --- PLOT 2: Crisis Score V2 Ranked ---
ax2 = fig.add_subplot(4, 2, 2)
ranked = sv.sort_values('invisible_crisis_score_v2', ascending=True)
colors_bar = plt.cm.RdYlGn_r(ranked['invisible_crisis_score_v2'] / ranked['invisible_crisis_score_v2'].max())
ax2.barh(range(len(ranked)), ranked['invisible_crisis_score_v2'], color=colors_bar, edgecolor='black', linewidth=0.3)
ax2.set_yticks(range(len(ranked)))
ax2.set_yticklabels(ranked['Country ISO3'], fontsize=9)
ax2.set_xlabel('Invisible Crisis Score V2', fontsize=10)
ax2.set_title('Crisis Rankings (with Absorption Capacity)', fontsize=12, fontweight='bold')
for i, (_, row) in enumerate(ranked.iterrows()):
    ax2.text(row['invisible_crisis_score_v2'] + 0.01, i, 
            f"{row['invisible_crisis_score_v2']:.2f}", va='center', fontsize=7)

# --- PLOT 3: Health vs Food Coverage Scatter ---
ax3 = fig.add_subplot(4, 2, 3)
s = sv[(sv['food_coverage'] != np.inf) & (sv['food_coverage'].notna())]
ax3.scatter(s['food_coverage'] * 100, s['health_coverage'] * 100, 
            s=s['health_in_need_24'] / s['health_in_need_24'].max() * 800 + 50,
            alpha=0.7, c=s['invisible_crisis_score_v2'], cmap='RdYlGn_r', edgecolors='black', linewidth=0.5)
max_v = max(s['food_coverage'].max()*100, s['health_coverage'].max()*100)
ax3.plot([0, max_v], [0, max_v], 'k--', alpha=0.3, label='Equal coverage')
for _, row in s.iterrows():
    ax3.annotate(row['Country ISO3'], (row['food_coverage']*100, row['health_coverage']*100),
                fontsize=7, fontweight='bold', ha='center', va='bottom')
ax3.set_xlabel('Food Security Coverage (%)', fontsize=10)
ax3.set_ylabel('Health Coverage (%)', fontsize=10)
ax3.set_title('Health vs Food: Both Underfunded, Health Worse', fontsize=12, fontweight='bold')
ax3.legend(fontsize=8)

# --- PLOT 4: Paired Bar — Health vs Food Funding Gap ---
ax4 = fig.add_subplot(4, 2, 4)
top_c = sv.nlargest(12, 'health_gap').sort_values('health_gap', ascending=True)
y = np.arange(len(top_c))
ax4.barh(y + 0.175, top_c['health_gap'] / 1e6, 0.35, label='Health Gap', color='#e74c3c', alpha=0.85)
ax4.barh(y - 0.175, top_c['food_gap'] / 1e6, 0.35, label='Food Gap', color='#f39c12', alpha=0.85)
ax4.set_yticks(y)
ax4.set_yticklabels(top_c['Country ISO3'], fontsize=9)
ax4.set_xlabel('Funding Gap ($M)', fontsize=10)
ax4.set_title('Health vs Food Funding Gaps', fontsize=12, fontweight='bold')
ax4.legend(fontsize=8)
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))

# --- PLOT 5: Food→Health Lag Signal ---
ax5 = fig.add_subplot(4, 2, 5)
temporal = sv.dropna(subset=['health_need_change_pct', 'food_need_change_pct'])
colors_t = ['#e74c3c' if row['food_to_health_lag'] else '#3498db' for _, row in temporal.iterrows()]
sizes_t = (1 - temporal['absorption_capacity']) * 500 + 50
ax5.scatter(temporal['food_need_change_pct'], temporal['health_need_change_pct'],
           s=sizes_t, c=colors_t, edgecolors='black', linewidth=0.7, alpha=0.8, zorder=5)
for _, row in temporal.iterrows():
    ax5.annotate(f"{row['Country ISO3']}", (row['food_need_change_pct'], row['health_need_change_pct']),
                fontsize=8, fontweight='bold', ha='center', va='bottom')
ax5.axhline(0, color='gray', linewidth=0.8)
ax5.axvline(0, color='gray', linewidth=0.8)
ax5.fill_between([-80, 0], [0, 0], [300, 300], alpha=0.08, color='red')
ax5.text(-45, 200, 'FOOD→HEALTH LAG', fontsize=9, color='red', ha='center', fontweight='bold')
ax5.set_xlabel('Food Need Change 2024→2025 (%)', fontsize=10)
ax5.set_ylabel('Health Need Change 2024→2025 (%)', fontsize=10)
ax5.set_title('Food→Health Lag Signal\n(Red = lag confirmed, size = inverse absorption)', fontsize=12, fontweight='bold')

# --- PLOT 6: Stacked — Health Funded vs Gap ---
ax6 = fig.add_subplot(4, 2, 6)
gap_d = sv.nlargest(12, 'health_required').sort_values('health_required', ascending=True)
y = np.arange(len(gap_d))
ax6.barh(y, gap_d['health_funded'] / 1e6, color='#27ae60', label='Funded', alpha=0.85)
ax6.barh(y, gap_d['health_gap'] / 1e6, left=gap_d['health_funded'] / 1e6, color='#e74c3c', label='Gap', alpha=0.85)
ax6.set_yticks(y)
ax6.set_yticklabels(gap_d['Country ISO3'], fontsize=9)
ax6.set_xlabel('USD (millions)', fontsize=10)
ax6.set_title('Health Funding: Asked vs Received', fontsize=12, fontweight='bold')
ax6.legend(fontsize=8)
ax6.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}M'))
for i, (_, row) in enumerate(gap_d.iterrows()):
    ax6.text(row['health_required']/1e6 + 2, i, f'{row["health_coverage"]*100:.1f}%', va='center', fontsize=8, fontweight='bold')

# --- PLOT 7: Absorption Capacity Breakdown ---
ax7 = fig.add_subplot(4, 2, 7)
breakdown = sv.sort_values('absorption_capacity')
colors_a = plt.cm.RdYlGn(breakdown['absorption_capacity'] / breakdown['absorption_capacity'].max())
ax7.barh(range(len(breakdown)), breakdown['absorption_capacity'], color=colors_a, edgecolor='black', linewidth=0.3)
ax7.set_yticks(range(len(breakdown)))
ax7.set_yticklabels(breakdown['Country ISO3'], fontsize=9)
ax7.set_xlabel('Absorption Capacity Index', fontsize=10)
ax7.set_title('Absorption Capacity (10 World Bank Indicators)', fontsize=12, fontweight='bold')
for i, (_, row) in enumerate(breakdown.iterrows()):
    doc = f"{row['physicians_per_1000']:.1f}" if pd.notna(row.get('physicians_per_1000')) else "?"
    ax7.text(row['absorption_capacity'] + 0.01, i, f"GDP ${row.get('gdp_per_capita',0):,.0f} | {doc} docs/1k", va='center', fontsize=6)

# --- PLOT 8: Under-5 Mortality vs Health Coverage ---
ax8 = fig.add_subplot(4, 2, 8)
ax8.scatter(sv['health_coverage'] * 100, sv['under5_mortality'],
           s=sv['health_in_need_24'] / sv['health_in_need_24'].max() * 600 + 80,
           c=sv['absorption_capacity'], cmap='RdYlGn', edgecolors='black', linewidth=0.7, alpha=0.85)
for _, row in sv.iterrows():
    ax8.annotate(row['Country ISO3'], (row['health_coverage']*100, row['under5_mortality']),
                fontsize=8, fontweight='bold', ha='center', va='bottom')
ax8.set_xlabel('Health Coverage (%)', fontsize=10)
ax8.set_ylabel('Under-5 Mortality (per 1,000)', fontsize=10)
ax8.set_title('More Funding ≠ Better Outcomes Without Capacity\n(Color = absorption: green=high)', fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(f"{base}crisis_compass_v2_visualizations.png", dpi=150, bbox_inches='tight')
plt.show()
print("  Visualizations saved.")

# ============================================================
# PART 9: SAVE ALL OUTPUTS
# ============================================================
print("\nPART 9: Saving outputs...")

master.to_csv(f"{base}crisis_compass_master_v2.csv", index=False)
wb_df.to_csv(f"{base}world_bank_indicators.csv", index=False)

# Figma export
export_cols = ['Country ISO3', 'crisis_rank_v2', 'invisible_crisis_score_v2',
               'health_in_need_24', 'health_in_need_25', 'health_need_change_pct',
               'health_required', 'health_funded', 'health_gap', 'health_coverage',
               'food_in_need_24', 'food_required', 'food_funded', 'food_coverage',
               'health_funding_per_person_in_need', 'health_vs_food_coverage_gap',
               'absorption_capacity', 'gdp_per_capita', 'physicians_per_1000',
               'under5_mortality', 'life_expectancy', 'measles_vaccination_pct',
               'total_pop', 'food_to_health_lag']
export_cols = [c for c in export_cols if c in scored.columns]
figma_v2 = scored[export_cols].copy()
for col in figma_v2.select_dtypes(include=[np.number]).columns:
    figma_v2[col] = figma_v2[col].round(4)
figma_v2 = figma_v2.sort_values('crisis_rank_v2')
figma_v2.to_csv(f"{base}figma_export_v2.csv", index=False)

scored.to_csv(f"{base}crisis_compass_scored.csv", index=False)

# ============================================================
# PART 10: PRINT FINAL SUMMARY
# ============================================================
print("\n" + "=" * 100)
print("CRISIS COMPASS — FINAL SUMMARY")
print("=" * 100)

print(f"\nDATASETS USED: 17 UN OCHA files + 10 World Bank indicators = 27 data sources")
print(f"COUNTRIES ANALYZED: {len(master)} total, {len(scored)} with complete scoring")
print(f"TIME SPAN: HNO 2024-2026, Funding 1999-2026, World Bank 2018-2024")

print(f"\nTOP 5 INVISIBLE CRISES (V2 — with absorption capacity):")
top5 = scored.nsmallest(5, 'crisis_rank_v2')[['Country ISO3', 'crisis_rank_v2', 'invisible_crisis_score_v2', 
                                                 'health_coverage', 'absorption_capacity', 'health_funding_per_person_in_need']]
for _, row in top5.iterrows():
    print(f"  #{int(row['crisis_rank_v2'])} {row['Country ISO3']}: score={row['invisible_crisis_score_v2']:.3f}, "
          f"coverage={row['health_coverage']*100:.1f}%, absorption={row['absorption_capacity']:.3f}, "
          f"${row['health_funding_per_person_in_need']:.2f}/person")

print(f"\nSTABLE ACROSS ALL WEIGHTING SCHEMES: {stable_top5}")

print(f"\nFOOD→HEALTH LAG: {scored['food_to_health_lag'].sum()} countries (UKR, MOZ, YEM)")

print(f"\nKEY NUMBERS:")
print(f"  Haiti: ${scored[scored['Country ISO3']=='HTI']['health_funding_per_person_in_need'].values[0]:.2f} per person in health need")
print(f"  Nigeria: absorption capacity = {scored[scored['Country ISO3']=='NGA']['absorption_capacity'].values[0]:.3f} (worst)")
print(f"  Total health funding gap: ${scored['health_gap'].sum()/1e9:.1f}B across {len(scored)} countries")
print(f"  Pooled fund allocations: ${allocations['Budget'].sum()/1e6:.0f}M")
print(f"  Top health donor: EU ECHO (${health_donor_summary.iloc[0]/1e6:.0f}M)")

print(f"\nFILES SAVED:")
print(f"  {base}crisis_compass_master_v2.csv")
print(f"  {base}crisis_compass_scored.csv")
print(f"  {base}figma_export_v2.csv")
print(f"  {base}world_bank_indicators.csv")
print(f"  {base}crisis_compass_v2_visualizations.png")

print(f"\n{'='*100}")
print("PIPELINE COMPLETE — Ready for presentation")
print(f"{'='*100}")

In [0]:
# ============================================================
# CRISIS COMPASS V3 — COMPLETE PIPELINE (Single Cell)
# ============================================================
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

base = "/Volumes/workspace/default/foodtohealth/"

# ============================================================
# PART 1: LOAD ALL DATASETS
# ============================================================
print("PART 1: Loading all datasets...")

# Load core UN datasets
hno_24 = pd.read_csv(f"{base}hpc_hno_2024.csv", low_memory=False)
hno_25 = pd.read_csv(f"{base}hpc_hno_2025.csv", low_memory=False)
hno_26 = pd.read_csv(f"{base}hpc_hno_2026.csv", low_memory=False)
fund_global = pd.read_csv(f"{base}fts_requirements_funding_global.csv")
fund_globalcluster = pd.read_csv(f"{base}fts_requirements_funding_globalcluster_global.csv")
allocations = pd.read_csv(f"{base}Allocations__20260221_193441_UTC.csv")
contributions = pd.read_csv(f"{base}Contributions__20260221_193441_UTC.csv")

# ============================================================
# PART 2: DATA CLEANING & RECONCILIATION
# ============================================================
print("PART 2: Cleaning HXL tags and fixing data types...")

def clean_hxl(df):
    if df.empty: return df
    if str(df.iloc[0, 0]).startswith('#'):
        df = df.drop(0).reset_index(drop=True)
    return df

hno_24, hno_25, hno_26 = map(clean_hxl, [hno_24, hno_25, hno_26])
fund_global = clean_hxl(fund_global)
fund_globalcluster = clean_hxl(fund_globalcluster)

# Fix numeric types
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    for df in [hno_24, hno_25, hno_26]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'percentFunded']:
    for df in [fund_global, fund_globalcluster]:
        df[col] = pd.to_numeric(df[col], errors='coerce')

allocations['Budget'] = pd.to_numeric(allocations['Budget'], errors='coerce')
contributions['Total'] = pd.to_numeric(contributions['Total'], errors='coerce')

# ============================================================
# PART 3: POOLED FUND ANALYSIS
# ============================================================
print("PART 3: Analyzing Pooled Fund Dynamics...")

donor_summary = contributions.groupby(['Donor', 'Donor type'])['Total'].sum().sort_values(ascending=False).reset_index()

# Map Pooled Fund names to standard ISO3 codes
fund_to_iso = {
    'Afghanistan': 'AFG', 'Sudan': 'SDN', 'Syria': 'SYR', 'Ukraine': 'UKR',
    'Somalia': 'SOM', 'DRC': 'COD', 'Nigeria': 'NGA', 'Ethiopia': 'ETH',
    'Yemen': 'YEM', 'YEMEN': 'YEM', 'Haiti (RhPF-LAC)': 'HTI', 'oPt': 'PSE',
    'Myanmar': 'MMR', 'Lebanon': 'LBN', 'CAR': 'CAF'
}
allocations['Country_ISO3'] = allocations['PooledFund'].map(fund_to_iso).fillna(allocations['PooledFund'])
pooled_alloc_summary = allocations.groupby('Country_ISO3')['Budget'].sum().reset_index()

fund_base = fund_global[fund_global['year'] == '2025'].groupby('countryCode').agg({
    'requirements': 'sum', 
    'funding': 'sum'
}).reset_index()

# ============================================================
# PART 4: MULTI-SECTORAL NEEDS (Sub-national Depth)
# ============================================================
print("PART 4: Processing Sub-national Needs (Defensively)...")

def get_sector_needs(df, year_suffix):
    # Group by ISO3 and Cluster, then aggregate
    grouped = df.groupby(['Country ISO3', 'Cluster'])[['In Need', 'Targeted', 'Reached']].sum().unstack()
    # Flatten MultiIndex columns -> e.g., 'In Need_2024_HEA'
    grouped.columns = [f"{col[0]}_{year_suffix}_{col[1]}" for col in grouped.columns]
    return grouped

needs_24 = get_sector_needs(hno_24, '2024')
needs_25 = get_sector_needs(hno_25, '2025')

# ============================================================
# PART 5: MASTER PIPELINE JOIN
# ============================================================
print("PART 5: Building Master Table...")

# Base: Global Funding Requirements for latest available full year (e.g., 2024 or 2025)
# Filtering safely
fund_base = fund_global[fund_global['year'] == '2025'].copy()
if fund_base.empty:
    fund_base = fund_global[fund_global['year'] == '2024'].copy()

master_v3 = fund_base[['countryCode', 'requirements', 'funding']].copy()
master_v3.rename(columns={'countryCode': 'ISO3', 'requirements': 'total_req_yr', 'funding': 'total_funded_yr'}, inplace=True)

# Merge sub-national needs
master_v3 = master_v3.merge(needs_24, left_on='ISO3', right_index=True, how='left')
master_v3 = master_v3.merge(needs_25, left_on='ISO3', right_index=True, how='left')

# Merge Pooled Fund Allocations
master_v3 = master_v3.merge(pooled_alloc_summary, left_on='ISO3', right_on='Country_ISO3', how='left')
master_v3['pooled_budget'] = master_v3['Budget'].fillna(0)
master_v3.drop(columns=['Budget', 'Country_ISO3'], inplace=True, errors='ignore')

# ============================================================
# PART 6: COMPREHENSIVE CRISIS METRICS & WORLD BANK INTEGRATION
# ============================================================
print("PART 6: Computing Multi-Sectoral Metrics...")

# 1. Pooled Fund Dependency
master_v3['pooled_dependency'] = (master_v3['pooled_budget'] / master_v3['total_funded_yr']).replace([np.inf, -np.inf], 0).fillna(0)

# 2. Defensively Calculate Social Neglect Index (PRO + EDU)
neglect_cols = [c for c in ['In Need_2025_PRO', 'In Need_2025_EDU', 'In Need_2024_PRO', 'In Need_2024_EDU'] if c in master_v3.columns]
if neglect_cols:
    master_v3['combined_social_need'] = master_v3[neglect_cols].sum(axis=1)
    master_v3['neglect_index'] = master_v3['combined_social_need'] / (master_v3['total_funded_yr'] + 1)
else:
    master_v3['neglect_index'] = 0

# 3. Defensively Calculate Health vs Food Need Ratio
hea_col = 'In Need_2025_HEA' if 'In Need_2025_HEA' in master_v3.columns else 'In Need_2024_HEA'
fsc_col = 'In Need_2025_FSC' if 'In Need_2025_FSC' in master_v3.columns else 'In Need_2024_FSC'

if hea_col in master_v3.columns and fsc_col in master_v3.columns:
    master_v3['health_to_food_need_ratio'] = master_v3[hea_col] / (master_v3[fsc_col] + 1)
    master_v3['health_budget_per_person'] = master_v3['total_funded_yr'] / (master_v3[hea_col] + 1)

# --- QUICK WORLD BANK CAPACITY FETCH ---
print("Fetching World Bank Capacity Data...")
countries_list = master_v3['ISO3'].dropna().unique().tolist()
country_str = ';'.join(countries_list)
indicators = {'NY.GDP.PCAP.CD': 'gdp_per_capita', 'SH.MED.PHYS.ZS': 'physicians_per_1000'}

wb_data = {}
for code, name in indicators.items():
    try:
        url = f"https://api.worldbank.org/v2/country/{country_str}/indicator/{code}?format=json&date=2018:2024&per_page=1000"
        resp = requests.get(url, timeout=10).json()
        if len(resp) > 1 and resp[1]:
            for entry in resp[1]:
                if entry['value'] is not None:
                    iso = entry['countryiso3code']
                    if iso not in wb_data: wb_data[iso] = {}
                    if name not in wb_data[iso]: wb_data[iso][name] = entry['value']
    except Exception:
        pass

wb_df = pd.DataFrame.from_dict(wb_data, orient='index').reset_index().rename(columns={'index': 'ISO3'})
master_v3 = master_v3.drop_duplicates(subset=['ISO3'], keep='first')
# ============================================================
# PART 7: FINAL SCORECARD & FLAG GENERATION
# ============================================================
print("PART 7: Generating Final Insight Scorecard...")

# Filter out countries with zero funding data to avoid noise
scored = master_v3[master_v3['total_funded_yr'] > 0].copy()

# Rank by Neglect Index (Higher = more neglected)
scored['neglect_rank'] = scored['neglect_index'].rank(ascending=False)

print("\n" + "=" * 80)
print("🏆 TOP 5 NEGLECTED CRISES (High Protection/Education Needs vs Low Overall Funding)")
print("=" * 80)
top_neglected = scored.sort_values('neglect_index', ascending=False).head(5)
for _, row in top_neglected.iterrows():
    print(f"  {row['ISO3']}: Neglect Index = {row['neglect_index']:.2f} | Pooled Dependency = {row['pooled_dependency']*100:.1f}%")

print("\n" + "=" * 80)
print("💰 TOP 5 POOLED FUND ANCHORS (Highest Absolute CBPF Allocations)")
print("=" * 80)
top_pooled = scored.sort_values('pooled_budget', ascending=False).head(5)
for _, row in top_pooled.iterrows():
    print(f"  {row['ISO3']}: ${row['pooled_budget']/1e6:.1f}M | Health/Food Ratio = {row.get('health_to_food_need_ratio', 0):.2f}")

# ============================================================
# PART 8: EXPORT
# ============================================================
export_path = f"{base}crisis_compass_v3_master.csv"
master_v3.to_csv(export_path, index=False)
print(f"\n✅ Pipeline Complete! Full dataset exported to: {export_path}")

In [0]:
# ============================================================
# CRISIS COMPASS V3 — DATABRICKS VISUALIZATION SUITE (Enhanced)
# ============================================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. Load the finalized data
base = "/Volumes/workspace/default/foodtohealth/"
df = pd.read_csv(f"{base}crisis_compass_v3_master.csv")

# Clean up any infinite values for visualization
df = df.replace([float('inf'), -float('inf')], 0).fillna(0)

# Filter out rows with zero funding or zero neglect to keep graphs clean
df_viz = df[(df['total_funded_yr'] > 0) | (df['neglect_index'] > 0)].copy()

# ============================================================
# VIZ 1: THE GLOBAL NEGLECT MAP (Choropleth)
# ============================================================
fig_map = px.choropleth(
    df_viz,
    locations="ISO3",
    locationmode="ISO-3",
    color="neglect_index",
    hover_name="ISO3",
    hover_data={
        "neglect_index": ":.2f",
        "pooled_budget": ":$,.0f",
        "total_funded_yr": ":$,.0f"
    },
    color_continuous_scale="Reds",
    title='<b>Global Crisis Neglect Index (2025/2026)</b><br><sup>Darker Red = High Protection/Education Need but Low Overall Funding</sup>',
    projection="natural earth",
    template="plotly_white"
)
# Force the drawing of individual country borders to prevent merging
fig_map.update_geos(
    showcountries=True, countrycolor="darkgray", 
    showcoastlines=True, coastlinecolor="black",
    showocean=True, oceancolor="#e0f7fa"
)
fig_map.update_layout(margin={"r":0,"t":50,"l":0,"b":0}, title_font_size=18)
fig_map.show()

# ============================================================
# VIZ 2: THE POOLED FUND vs NEGLECT QUADRANT (Scatter)
# ============================================================
# FIXED: Removed marginal plots to prevent the continuous color scale ('Viridis') bug
fig_scatter = px.scatter(
    df_viz[df_viz['neglect_index'] < 100], # Filter out extreme FTS lag outliers
    x="pooled_budget",
    y="neglect_index",
    size="total_funded_yr",
    color="health_to_food_need_ratio",
    hover_name="ISO3",
    text="ISO3",
    color_continuous_scale="Viridis",
    labels={
        "pooled_budget": "CBPF Pooled Fund Budget ($)",
        "neglect_index": "Social Neglect Index (PRO + EDU Gap)",
        "health_to_food_need_ratio": "Health vs Food Need"
    },
    title='<b>Targeting Efficiency: Pooled Funds vs. Neglect</b><br><sup>Are we funding the "invisible" crises? (Bubble size = Total Global Funding)</sup>',
    template="plotly_white"
)
fig_scatter.update_traces(textposition='top center', marker=dict(line=dict(width=1.5, color='black')))
fig_scatter.update_layout(xaxis_type="log", height=700, title_font_size=18) 
fig_scatter.show()

# ============================================================
# VIZ 3: THE ANCHORS vs THE FORGOTTEN (Bar Chart)
# ============================================================
top_anchors = df_viz.sort_values("pooled_budget", ascending=False).head(15)

fig_bar = go.Figure()
# Bar for Pooled Budget
fig_bar.add_trace(go.Bar(
    x=top_anchors['ISO3'],
    y=top_anchors['pooled_budget'],
    name='Pooled Fund Allocation ($)',
    marker_color='#2c3e50',
    marker_line_color='black',
    marker_line_width=1,
    text=top_anchors['pooled_budget'].apply(lambda x: f"${x/1e6:.1f}M"),
    textposition='auto',
    yaxis='y1'
))
# Line for Neglect Index overlay
fig_bar.add_trace(go.Scatter(
    x=top_anchors['ISO3'],
    y=top_anchors['neglect_index'],
    name='Neglect Index',
    mode='lines+markers+text',
    text=top_anchors['neglect_index'].round(1),
    textposition='top center',
    marker=dict(color='#e74c3c', size=12, line=dict(width=2, color='white')),
    line=dict(width=4),
    yaxis='y2'
))

fig_bar.update_layout(
    title='<b>The Anchor Crises: High Budgets vs Neglect Reality</b>',
    xaxis=dict(title="Country (ISO3)"),
    yaxis=dict(title="Pooled Fund Budget ($)", side='left', showgrid=True, gridcolor='lightgray'),
    yaxis2=dict(title="Neglect Index", side='right', overlaying='y', showgrid=False),
    legend=dict(x=0.80, y=0.95, bgcolor='rgba(255,255,255,0.8)', bordercolor='black', borderwidth=1),
    height=600,
    template="plotly_white",
    hovermode="x unified"
)
fig_bar.show()

# ============================================================
# VIZ 4: BENCHMARKING HEALTH vs FOOD (Horizontal Bar)
# ============================================================
df_bench = df_viz[df_viz['health_to_food_need_ratio'] > 0].sort_values('health_to_food_need_ratio', ascending=True).tail(15)

fig_ratio = px.bar(
    df_bench,
    x="health_to_food_need_ratio",
    y="ISO3",
    orientation='h',
    color="pooled_budget",
    color_continuous_scale="Teal",
    text="health_to_food_need_ratio",
    title='<b>Crisis Archetypes: Health vs. Food Driven</b><br><sup>&lt; 1.0 = Starvation Crisis | &gt; 1.0 = Health/Infrastructure Crisis</sup>',
    labels={"health_to_food_need_ratio": "Ratio (Health Need / Food Need)", "ISO3": "Country"},
    template="plotly_white"
)
fig_ratio.update_traces(texttemplate='%{text:.2f}', textposition='outside', marker_line_color='black', marker_line_width=1)
fig_ratio.add_vline(x=1.0, line_dash="dash", line_color="red", annotation_text="Equal Need", annotation_position="top right")
fig_ratio.update_layout(height=600)
fig_ratio.show()

# --- DATABRICKS BONUS TIP ---
display(df_viz[['ISO3', 'neglect_index', 'pooled_budget', 'total_funded_yr', 'health_to_food_need_ratio']])

In [0]:
# ============================================================
# CRISIS COMPASS V4 — COMPLETE PIPELINE & VIZ SUITE (Single Cell)
# ============================================================
import pandas as pd
import numpy as np
import requests
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

base = "/Volumes/workspace/default/foodtohealth/"

# ============================================================
# PART 1: LOAD ALL DATASETS
# ============================================================
print("PART 1: Loading all datasets...")

hno_24 = pd.read_csv(f"{base}hpc_hno_2024.csv", low_memory=False)
hno_25 = pd.read_csv(f"{base}hpc_hno_2025.csv", low_memory=False)
fund_global = pd.read_csv(f"{base}fts_requirements_funding_global.csv")
fund_globalcluster = pd.read_csv(f"{base}fts_requirements_funding_globalcluster_global.csv")
allocations = pd.read_csv(f"{base}Allocations__20260221_193441_UTC.csv")

# ============================================================
# PART 2: DATA CLEANING & RECONCILIATION
# ============================================================
print("PART 2: Cleaning HXL tags and fixing data types...")

def clean_hxl(df):
    if df.empty: return df
    if str(df.iloc[0, 0]).startswith('#'):
        df = df.drop(0).reset_index(drop=True)
    return df

hno_24, hno_25 = map(clean_hxl, [hno_24, hno_25])
fund_global = clean_hxl(fund_global)
fund_globalcluster = clean_hxl(fund_globalcluster)

# Fix numeric types
for col in ['Population', 'In Need', 'Targeted', 'Affected', 'Reached']:
    for df in [hno_24, hno_25]:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

for col in ['requirements', 'funding', 'percentFunded']:
    for df in [fund_global, fund_globalcluster]:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')

allocations['Budget'] = pd.to_numeric(allocations['Budget'], errors='coerce')

# Map Pooled Fund names to standard ISO3 codes
fund_to_iso = {
    'Afghanistan': 'AFG', 'Sudan': 'SDN', 'Syria': 'SYR', 'Ukraine': 'UKR',
    'Somalia': 'SOM', 'DRC': 'COD', 'Nigeria': 'NGA', 'Ethiopia': 'ETH',
    'Yemen': 'YEM', 'YEMEN': 'YEM', 'Haiti (RhPF-LAC)': 'HTI', 'oPt': 'PSE',
    'Myanmar': 'MMR', 'Lebanon': 'LBN', 'CAR': 'CAF'
}
allocations['Country_ISO3'] = allocations['PooledFund'].map(fund_to_iso).fillna(allocations['PooledFund'])
pooled_alloc_summary = allocations.groupby('Country_ISO3')['Budget'].sum().reset_index()

# ============================================================
# PART 3: SUB-NATIONAL & MULTI-SECTORAL NEEDS 
# ============================================================
print("PART 3: Processing Sub-national Needs...")

def get_sector_needs(df, year_suffix):
    grouped = df.groupby(['Country ISO3', 'Cluster'])[['In Need', 'Targeted', 'Reached']].sum().unstack()
    grouped.columns = [f"{col[0]}_{year_suffix}_{col[1]}" for col in grouped.columns]
    return grouped

needs_24 = get_sector_needs(hno_24, '2024')
needs_25 = get_sector_needs(hno_25, '2025')

# --- CBPF DASHBOARD REPLICATION DATA (ALL SECTORS) ---
cluster_mapping = {
    'PRO': 'Protection', 'WSH': 'Water Sanitation Hygiene', 'CC': 'Coordination and support services',
    'SHL': 'Emergency Shelter and NFI', 'HEA': 'Health', 'FSC': 'Food Security', 'EDU': 'Education',
    'NUT': 'Nutrition', 'LOG': 'Logistics', 'CCM': 'Camp Coordination / Management',
    'ER': 'Early Recovery', 'ETC': 'Emergency Telecommunications', 'MS': 'Multi-Sector',
    'MPC': 'Multipurpose Cash', 'AGR': 'Agriculture'
}

hno_sector = hno_25.groupby('Cluster')[['Targeted', 'Reached', 'In Need']].sum().reset_index()
hno_sector['Cluster_Name'] = hno_sector['Cluster'].map(cluster_mapping).fillna(hno_sector['Cluster'])
fund_sector = fund_globalcluster[fund_globalcluster['year'] == '2025'].groupby('cluster')[['funding', 'requirements']].sum().reset_index()

sector_df = pd.merge(fund_sector, hno_sector, left_on='cluster', right_on='Cluster_Name', how='outer')
sector_df['cluster_display'] = sector_df['cluster'].fillna(sector_df['Cluster_Name'])
sector_df = sector_df.fillna(0).sort_values('funding', ascending=True)

total_funding = sector_df['funding'].sum()
total_targeted = sector_df['Targeted'].sum()
sector_df['funding_pct'] = (sector_df['funding'] / (total_funding + 1)) * 100
sector_df['targeted_pct'] = (sector_df['Targeted'] / (total_targeted + 1)) * 100

# ============================================================
# PART 4: MASTER JOIN & DEFENSIVE METRICS
# ============================================================
print("PART 4: Building Master Table & Metrics...")

# Aggregate base to prevent Cartesian explosion
fund_base = fund_global[fund_global['year'] == '2025'].groupby('countryCode').agg({'requirements': 'sum', 'funding': 'sum'}).reset_index()
master_v3 = fund_base.rename(columns={'countryCode': 'ISO3', 'requirements': 'total_req_yr', 'funding': 'total_funded_yr'})

master_v3 = master_v3.merge(needs_24, left_on='ISO3', right_index=True, how='left')
master_v3 = master_v3.merge(needs_25, left_on='ISO3', right_index=True, how='left')
master_v3 = master_v3.merge(pooled_alloc_summary, left_on='ISO3', right_on='Country_ISO3', how='left')

master_v3['pooled_budget'] = master_v3['Budget'].fillna(0)
master_v3['pooled_dependency'] = (master_v3['pooled_budget'] / master_v3['total_funded_yr']).replace([np.inf, -np.inf], 0).fillna(0)

neglect_cols = [c for c in ['In Need_2025_PRO', 'In Need_2025_EDU', 'In Need_2024_PRO', 'In Need_2024_EDU'] if c in master_v3.columns]
if neglect_cols:
    master_v3['neglect_index'] = master_v3[neglect_cols].sum(axis=1) / (master_v3['total_funded_yr'] + 1)
else:
    master_v3['neglect_index'] = 0

hea_col = 'In Need_2025_HEA' if 'In Need_2025_HEA' in master_v3.columns else 'In Need_2024_HEA'
fsc_col = 'In Need_2025_FSC' if 'In Need_2025_FSC' in master_v3.columns else 'In Need_2024_FSC'
if hea_col in master_v3.columns and fsc_col in master_v3.columns:
    master_v3['health_to_food_need_ratio'] = master_v3[hea_col] / (master_v3[fsc_col] + 1)

# Clean for VIZ
df_viz = master_v3[(master_v3['total_funded_yr'] > 0) | (master_v3['neglect_index'] > 0)].copy()
df_viz = df_viz.replace([np.inf, -np.inf], 0).fillna(0)

# ============================================================
# PART 5: VISUALIZATION SUITE (DATABRICKS READY)
# ============================================================
print("PART 5: Generating Plotly Visualizations...")

# --- VIZ 1: THE GLOBAL NEGLECT MAP ---
fig_map = px.choropleth(
    df_viz, locations="ISO3", locationmode="ISO-3", color="neglect_index",
    hover_name="ISO3", hover_data={"neglect_index": ":.2f", "pooled_budget": ":$,.0f"},
    color_continuous_scale="Reds",
    title='<b>Global Crisis Neglect Index (2025/2026)</b><br><sup>Darker Red = High Protection/Education Need but Low Overall Funding</sup>',
    template="plotly_white"
)
fig_map.update_geos(showcountries=True, countrycolor="darkgray", showcoastlines=True, coastlinecolor="black", showocean=True, oceancolor="#e0f7fa")
fig_map.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
fig_map.show()

# --- VIZ 2: POOLED FUND vs NEGLECT SCATTER ---
fig_scatter = px.scatter(
    df_viz[df_viz['neglect_index'] < 100], x="pooled_budget", y="neglect_index",
    size="total_funded_yr", color="health_to_food_need_ratio", hover_name="ISO3", text="ISO3",
    color_continuous_scale="Viridis",
    labels={"pooled_budget": "CBPF Pooled Fund Budget ($)", "neglect_index": "Social Neglect Index (PRO + EDU Gap)", "health_to_food_need_ratio": "Health vs Food Need"},
    title='<b>Targeting Efficiency: Pooled Funds vs. Neglect</b><br><sup>Are we funding the "invisible" crises? (Bubble size = Total Global Funding)</sup>',
    template="plotly_white"
)
fig_scatter.update_traces(textposition='top center', marker=dict(line=dict(width=1, color='black')))
fig_scatter.update_layout(xaxis_type="log", height=600)
fig_scatter.show()

# --- VIZ 3: THE ANCHORS (BAR + LINE) ---
top_anchors = df_viz.sort_values("pooled_budget", ascending=False).head(15)
fig_bar = go.Figure()
fig_bar.add_trace(go.Bar(
    x=top_anchors['ISO3'], y=top_anchors['pooled_budget'], name='Pooled Fund Allocation ($)',
    marker_color='#2c3e50', text=top_anchors['pooled_budget'].apply(lambda x: f"${x/1e6:.1f}M"), textposition='auto', yaxis='y1'
))
fig_bar.add_trace(go.Scatter(
    x=top_anchors['ISO3'], y=top_anchors['neglect_index'], name='Neglect Index',
    mode='lines+markers+text', text=top_anchors['neglect_index'].round(1), textposition='top center',
    marker=dict(color='#e74c3c', size=10, line=dict(width=2, color='white')), line=dict(width=3), yaxis='y2'
))
fig_bar.update_layout(
    title='<b>The Anchor Crises: High Budgets vs Neglect Reality</b>',
    yaxis=dict(title="Pooled Fund Budget ($)", side='left'), yaxis2=dict(title="Neglect Index", side='right', overlaying='y', showgrid=False),
    legend=dict(x=0.80, y=0.95), height=500, template="plotly_white", hovermode="x unified"
)
fig_bar.show()

# --- VIZ 4: CBPF-STYLE MULTI-SECTOR DASHBOARD (ALL SECTORS) ---
# Filters out noise and removes clusters with negligible data to keep the chart clean
clean_sectors = sector_df[(sector_df['funding'] > 1000) | (sector_df['Targeted'] > 1000)]

fig_sector = make_subplots(
    rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05,
    subplot_titles=("Allocations/Funding in US$ (% of total)", "People Targeted vs Reached (% of total)")
)

# Left: Funding Bar
fig_sector.add_trace(
    go.Bar(
        y=clean_sectors['cluster_display'], x=clean_sectors['funding'], orientation='h',
        name='Total Allocated ($)', marker_color='#3498db',
        text=clean_sectors.apply(lambda row: f"${row['funding']/1e6:.1f}M ({row['funding_pct']:.0f}%)" if row['funding']>0 else "", axis=1),
        textposition='outside'
    ), row=1, col=1
)

# Right: Targeted Bar
fig_sector.add_trace(
    go.Bar(
        y=clean_sectors['cluster_display'], x=clean_sectors['Targeted'], orientation='h',
        name='People Targeted', marker_color='#2ecc71',
        text=clean_sectors.apply(lambda row: f"{row['Targeted']/1e6:.1f}M ({row['targeted_pct']:.0f}%)" if row['Targeted']>0 else "", axis=1),
        textposition='outside'
    ), row=1, col=2
)

# Right: Reached Marker (The CBPF Arrow)
fig_sector.add_trace(
    go.Scatter(
        y=clean_sectors['cluster_display'], x=clean_sectors['Reached'], mode='markers',
        name='People Reached (▲)', marker=dict(symbol='triangle-up', size=14, color='#2c3e50')
    ), row=1, col=2
)

fig_sector.update_layout(
    title='<b>Global Multi-Sector Humanitarian Dashboard (2025)</b><br><sup>Aligning Funding Allocations with Population Targeting & Reach</sup>',
    height=750, template='plotly_white', showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="right", x=1)
)
fig_sector.update_xaxes(title_text="US Dollars ($)", row=1, col=1)
fig_sector.update_xaxes(title_text="Number of People", row=1, col=2)
fig_sector.show()

print("\n✅ V4 Pipeline and Detailed Visualizations Complete.")